# AuraGateway preflight-v3 final exact-runtime offline compatibility verifier v5

Implements the accepted typed semantic boundary: raw subprocess observations are parsed and validated before terminal evidence projection. Preserves V4 saved version 341211001 as diagnostic evidence. No model loading, worker startup, requests, P5/P6, pilot, or benchmark execution. Execution remains separately authorized.


In [1]:
from __future__ import annotations

import hashlib
import importlib
import importlib.metadata
import json
import os
import re
import shutil
import subprocess
import sys
import time
import zipfile
from dataclasses import asdict, dataclass
from datetime import UTC, datetime
from enum import StrEnum
from pathlib import Path, PurePosixPath
from typing import Callable, Generic, TypeVar

NOTEBOOK_NAME = (
    "auragateway-preflight-v3-exact-runtime-offline-compatibility-v5"
)
REQUESTED_KAGGLE_TITLE = "ag-preflight-v3-final-offline-verifier-v5"
INPUT_DIRECTORY_NAME = (
    "auragateway_preflight_v3_exact_runtime_wheelhouse_v1"
)
OUTPUT_DIRECTORY_NAME = (
    "auragateway_preflight_v3_exact_runtime_offline_compatibility_evidence_v5"
)
EVIDENCE_ROOT = Path("/kaggle/working") / OUTPUT_DIRECTORY_NAME
OUTPUT_ZIP = Path("/kaggle/working") / f"{OUTPUT_DIRECTORY_NAME}.zip"
TARGET_ROOT = Path(
    "/kaggle/working/auragateway_preflight_v3_exact_runtime_target_v5"
)

EXPECTED_MATERIALIZER_SCRIPT_VERSION_ID = 341083505
EXPECTED_PACKAGE_COUNT = 196
EXPECTED_SHA_MANIFEST_ENTRY_COUNT = 200
EXPECTED_SHA_MANIFEST_WHEEL_COUNT = 196
EXPECTED_SHA_MANIFEST_CONTROL_COUNT = 4
EXPECTED_AUTHORITY_HOST_COUNT = 5
EXPECTED_TOTAL_WHEEL_BYTES = 6164913809

EXPECTED_RESOLUTION_LOCK_SHA256 = (
    "1294394ac476336b103b036d8654a49e4ae78c25c912ca5729cd94f982384f3c"
)
EXPECTED_CONTROL_HASHES = {
    "resolution_lock.json": (
        "1294394ac476336b103b036d8654a49e4ae78c25c912ca5729cd94f982384f3c"
    ),
    "requirements.lock.txt": (
        "cf5d773ef5c26f2e42a7afd76f0e466c21847169986f14fe5a7ac9ad02f0a3c3"
    ),
    "materialization.lock.txt": (
        "774461508794d804244b2f0dbff05e52fdccc8efbe19af8cfb8d0faedcb25339"
    ),
    "runtime_manifest.json": (
        "cb9c62321ea1651deac260126db75c39525e4ba711ee3708fe5f7a5b50ffd6ed"
    ),
    "sha256_manifest.json": (
        "00dbda4fd734cf94b6f5dfde2619f83ed6a4db7761a4c3c5ace6b0f1ebe63b08"
    ),
    "materialization_receipt.json": (
        "55bc8d078af9960d5f6a60bf7d9638820be9fdda0ee76754a9462d46eb053fe0"
    ),
}

EXPECTED_RUNTIME = {
    "python_major_minor": "3.12",
    "cuda_variant": "cu129",
    "torch_cuda_version": "12.9",
    "torch": "2.11.0+cu129",
    "torchaudio": "2.11.0+cu129",
    "torchvision": "0.26.0+cu129",
    "transformers": "5.14.1",
    "triton": "3.6.0",
    "vllm": "0.25.1+cu129",
}
EXPECTED_VLLM_MODULE_VERSION = "0.25.1"
REQUIRED_NATIVE_MODULE = "vllm._C_stable_libtorch"

PREDECESSOR_V4_SAVED_VERSION_ID = 341211001
PREDECESSOR_V4_FAILURE_CLASS = "DIAGNOSTIC_HARNESS_DEFECT"
PREDECESSOR_V4_FAILURE_CODE = (
    "EVIDENCE_REPRESENTATION_REUSED_AS_SEMANTIC_INPUT"
)
PREDECESSOR_V4_EVIDENCE_ZIP_SHA256 = (
    "94e73e06c2627c9c03fac85894654800e31fbd6f55b0c6157ea0d09097ef92c8"
)

PRODUCER_RECEIPT_EXPECTED = {
    "materialization_status": "PASSED_PENDING_REPOSITORY_ACCEPTANCE",
    "exact_resolution_lock_sha256": EXPECTED_RESOLUTION_LOCK_SHA256,
    "locked_package_count": 196,
    "downloaded_package_count": 196,
    "wheel_file_count": 196,
    "authority_host_count": 5,
    "observed_transport_redirect_event_count": 1,
    "total_wheel_bytes": 6164913809,
    "dependency_resolution_performed": False,
    "package_installation_performed": False,
    "model_loads_performed": 0,
    "model_requests_performed": 0,
    "benchmark_trajectories_performed": 0,
    "credentials_used": False,
    "customer_data_used": False,
    "external_spend": 0,
    "wheelhouse_materialized": True,
    "exact_runtime_materialized": False,
    "exact_runtime_offline_verified": False,
    "qualification_claimed": False,
    "sha256_manifest_sha256": (
        "00dbda4fd734cf94b6f5dfde2619f83ed6a4db7761a4c3c5ace6b0f1ebe63b08"
    ),
}
CONSUMER_CAPABILITY_POLICY = {
    "controlled_python_startup_required": True,
    "native_loader_provenance_required": True,
    "successful_native_import_alone_sufficient": False,
}
REAL_DRIVER_DIRECTORY = Path("/usr/local/nvidia/lib64")
PROHIBITED_LIBRARY_PATH_MARKERS = (
    "/usr/local/cuda/lib64/stubs",
    "/usr/local/cuda/compat",
)
GOVERNED_NATIVE_LIBRARY_BASENAMES = (
    "libtorch",
    "libc10",
    "libcudart",
    "libnvJitLink",
    "libcusparse",
    "libcublas",
    "libcufft",
    "libcurand",
    "libcusolver",
    "libnccl",
    "libnvrtc",
    "libcuda",
)

EXPECTED_TOP_LEVEL = frozenset(
    {
        "wheels",
        "resolution_lock.json",
        "requirements.lock.txt",
        "materialization.lock.txt",
        "runtime_manifest.json",
        "sha256_manifest.json",
        "materialization_receipt.json",
    }
)
EXPECTED_SHA_MANIFEST_CONTROL_PATHS = frozenset(
    {
        "resolution_lock.json",
        "requirements.lock.txt",
        "materialization.lock.txt",
        "runtime_manifest.json",
    }
)

REQUIRED_ROLES = (
    "input_validation",
    "base_python_runtime",
    "base_pip_import",
    "base_distribution_snapshot_before",
    "gpu_topology",
    "target_environment_creation",
    "target_runtime_identity_before_install",
    "base_pip_python_target_support",
    "offline_hash_locked_install_via_base_pip",
    "target_distribution_inventory",
    "target_dependency_check_via_base_pip",
    "controlled_python_startup",
    "target_native_inventory",
    "canonical_loader_environment",
    "python_runtime",
    "torch_family_runtime",
    "transformers_runtime",
    "triton_distribution",
    "vllm_distribution",
    "vllm_module",
    "native_linker_static_provenance",
    "vllm_native_extension",
    "native_runtime_provenance",
    "cuda_platform_capability",
    "base_distribution_snapshot_after",
)

MAX_EXCERPT = 12000
CHUNK_BYTES = 8 * 1024 * 1024
_SHA256_PATTERN = re.compile(r"^[0-9a-f]{64}$")


class ProbeStatus(StrEnum):
    PASSED = "PASSED"
    FAILED = "FAILED"
    BLOCKED_BY_UPSTREAM_FAILURE = "BLOCKED_BY_UPSTREAM_FAILURE"
    NOT_EXECUTED = "NOT_EXECUTED"


class FailureCode(StrEnum):
    SUBPROCESS_FAILED = "SUBPROCESS_FAILED"
    SUBPROCESS_TIMEOUT = "SUBPROCESS_TIMEOUT"
    SEMANTIC_PARSE_FAILED = "SEMANTIC_PARSE_FAILED"
    SEMANTIC_CONTRACT_FAILED = "SEMANTIC_CONTRACT_FAILED"
    INPUT_VALIDATION_FAILED = "INPUT_VALIDATION_FAILED"
    PROHIBITED_NATIVE_ORIGIN = "PROHIBITED_NATIVE_ORIGIN"
    UNKNOWN_NATIVE_ORIGIN = "UNKNOWN_NATIVE_ORIGIN"
    BLOCKED_BY_UPSTREAM_FAILURE = "BLOCKED_BY_UPSTREAM_FAILURE"


class NativeOriginClass(StrEnum):
    TARGET_OWNED = "TARGET_OWNED"
    PERMITTED_HOST_PLATFORM = "PERMITTED_HOST_PLATFORM"
    PROHIBITED_AMBIENT = "PROHIBITED_AMBIENT"
    UNKNOWN = "UNKNOWN"


@dataclass(frozen=True, slots=True)
class RawProbeExecution:
    command_role: str
    returncode: int | None
    timed_out: bool
    duration_ms: int
    started_at: str
    stdout: str
    stderr: str


@dataclass(frozen=True, slots=True)
class ProbeDecision:
    status: ProbeStatus
    failure_code: FailureCode | None = None
    detail: str = ""


@dataclass(frozen=True, slots=True)
class ProbeEvidenceRecord:
    schema_version: str
    command_role: str
    status: str
    failure_code: str | None
    dependencies: tuple[str, ...]
    started_at: str | None
    duration_ms: int
    returncode: int | None
    timed_out: bool
    stdout_excerpt: str
    stderr_excerpt: str
    detail: str

    def to_dict(self) -> dict[str, object]:
        payload = asdict(self)
        payload["dependencies"] = list(self.dependencies)
        return payload


ObservationT = TypeVar("ObservationT")


@dataclass(frozen=True, slots=True)
class ProbeOutcome(Generic[ObservationT]):
    observation: ObservationT | None
    decision: ProbeDecision
    evidence: ProbeEvidenceRecord


@dataclass(frozen=True, slots=True)
class InputValidationObservation:
    input_root: Path
    lock_records: tuple[dict[str, object], ...]


@dataclass(frozen=True, slots=True)
class PythonVersionObservation:
    python: str


@dataclass(frozen=True, slots=True)
class DistributionSnapshotObservation:
    count: int
    sha256: str


@dataclass(frozen=True, slots=True)
class GpuTopologyObservation:
    rows: tuple[str, ...]


@dataclass(frozen=True, slots=True)
class TargetIdentityObservation:
    python: str
    prefix_matches_expected: bool
    base_prefix_differs: bool
    user_site_enabled: bool
    purelib_within_prefix: bool
    platlib_within_prefix: bool
    system_site_packages_enabled: bool
    pip_present: bool


@dataclass(frozen=True, slots=True)
class DistributionInventoryObservation:
    items: tuple[tuple[str, str], ...]


@dataclass(frozen=True, slots=True)
class ControlledStartupObservation:
    prefix: Path
    base_prefix: Path
    no_site_flag: int
    user_site_enabled: bool
    target_site_present: bool
    external_package_paths: tuple[Path, ...]
    sitecustomize_file: str
    usercustomize_file: str
    pythonpath_present: bool
    pythonhome_present: bool
    ld_preload_present: bool
    python_no_user_site: str


@dataclass(frozen=True, slots=True)
class NativeFileObservation:
    path: Path
    sha256: str
    size_bytes: int


@dataclass(frozen=True, slots=True)
class NativeInventoryObservation:
    required: tuple[NativeFileObservation, ...]
    legacy_c_candidates: tuple[Path, ...]
    optional_candidates: tuple[Path, ...]


@dataclass(frozen=True, slots=True)
class CanonicalLoaderObservation:
    target_nvidia_library_count: int
    target_torch_library: Path
    real_driver_directory: Path
    ld_preload_absent: bool
    pythonpath_absent: bool
    pythonhome_absent: bool
    prohibited_path_present: bool


@dataclass(frozen=True, slots=True)
class TorchFamilyObservation:
    torch: str
    torchaudio: str
    torchvision: str
    torch_cuda_version: str
    cuda_available: bool
    cuda_device_count: int
    cuda_device_names: tuple[str, ...]


@dataclass(frozen=True, slots=True)
class VersionObservation:
    package: str
    version: str


@dataclass(frozen=True, slots=True)
class NativeLinkerObservation:
    unresolved_required_library: bool
    resolved_paths: tuple[Path, ...]


@dataclass(frozen=True, slots=True)
class NativeExtensionObservation:
    native_extension: str
    file: Path


@dataclass(frozen=True, slots=True)
class NativeRuntimeProvenanceObservation:
    native_module: str
    native_file: Path
    torch_file: Path
    vllm_file: Path
    cuda_available: bool
    loaded_paths: tuple[Path, ...]


@dataclass(frozen=True, slots=True)
class CudaPlatformObservation:
    cuda_available: bool
    cuda_device_count: int
    cuda_device_names: tuple[str, ...]
    required_native_module_loaded: bool


def canonical_json(payload: object) -> str:
    return json.dumps(
        payload,
        ensure_ascii=True,
        separators=(",", ":"),
        sort_keys=True,
    )


def write_json(path: Path, payload: object) -> None:
    path.write_text(
        canonical_json(payload) + "\n",
        encoding="utf-8",
        newline="\n",
    )


def streaming_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while True:
            chunk = handle.read(CHUNK_BYTES)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def normalize_name(value: str) -> str:
    return re.sub(r"[-_.]+", "-", value).lower()


def truncate_evidence_text(text: str, limit: int = MAX_EXCERPT) -> str:
    return text[-limit:]


def sanitize_evidence_text(
    text: str | bytes | None,
    *,
    limit: int = MAX_EXCERPT,
    replacements: dict[str, str] | None = None,
) -> str:
    if text is None:
        return ""
    decoded = (
        text.decode("utf-8", errors="replace")
        if isinstance(text, bytes)
        else text
    )
    evidence_replacements = (
        {
            "/kaggle/input": "<input>",
            "/kaggle/working": "<working>",
        }
        if replacements is None
        else dict(replacements)
    )
    home = os.environ.get("HOME")
    if home and home not in evidence_replacements:
        evidence_replacements[home] = "<home>"
    for source, replacement in evidence_replacements.items():
        decoded = decoded.replace(source, replacement)
    return truncate_evidence_text(decoded, limit)


def project_evidence(
    raw: RawProbeExecution,
    decision: ProbeDecision,
    *,
    dependencies: tuple[str, ...],
    excerpt_limit: int = MAX_EXCERPT,
    evidence_replacements: dict[str, str] | None = None,
) -> ProbeEvidenceRecord:
    return ProbeEvidenceRecord(
        schema_version="1.0.0",
        command_role=raw.command_role,
        status=decision.status.value,
        failure_code=(
            None
            if decision.failure_code is None
            else decision.failure_code.value
        ),
        dependencies=dependencies,
        started_at=raw.started_at,
        duration_ms=raw.duration_ms,
        returncode=raw.returncode,
        timed_out=raw.timed_out,
        stdout_excerpt=sanitize_evidence_text(
            raw.stdout,
            limit=excerpt_limit,
            replacements=evidence_replacements,
        ),
        stderr_excerpt=sanitize_evidence_text(
            raw.stderr,
            limit=excerpt_limit,
            replacements=evidence_replacements,
        ),
        detail=sanitize_evidence_text(
            decision.detail,
            limit=excerpt_limit,
            replacements=evidence_replacements,
        ),
    )


def project_local_evidence(
    role: str,
    decision: ProbeDecision,
    *,
    dependencies: tuple[str, ...],
    started_at: str | None,
    duration_ms: int,
    diagnostic_text: str = "",
) -> ProbeEvidenceRecord:
    raw = RawProbeExecution(
        command_role=role,
        returncode=None,
        timed_out=False,
        duration_ms=duration_ms,
        started_at=started_at or "",
        stdout="",
        stderr="",
    )
    evidence = project_evidence(
        raw,
        decision,
        dependencies=dependencies,
    )
    if diagnostic_text:
        return ProbeEvidenceRecord(
            **{
                **evidence.to_dict(),
                "dependencies": evidence.dependencies,
                "detail": sanitize_evidence_text(diagnostic_text),
            }
        )
    return evidence


def run_probe_raw(
    role: str,
    argv: list[str],
    *,
    timeout: float,
    environment: dict[str, str] | None = None,
) -> RawProbeExecution:
    started = time.monotonic()
    started_at = datetime.now(UTC).isoformat(timespec="seconds")
    probe_environment = (
        {**environment}
        if environment is not None
        else {
            **os.environ,
            "PIP_DISABLE_PIP_VERSION_CHECK": "1",
            "PIP_NO_INDEX": "1",
            "PIP_NO_CACHE_DIR": "1",
            "HF_HUB_OFFLINE": "1",
            "TRANSFORMERS_OFFLINE": "1",
        }
    )
    try:
        result = subprocess.run(
            argv,
            check=False,
            capture_output=True,
            text=True,
            timeout=timeout,
            env=probe_environment,
        )
    except subprocess.TimeoutExpired as exc:
        return RawProbeExecution(
            command_role=role,
            returncode=None,
            timed_out=True,
            duration_ms=int((time.monotonic() - started) * 1000),
            started_at=started_at,
            stdout=(
                ""
                if exc.stdout is None
                else (
                    exc.stdout.decode("utf-8", errors="replace")
                    if isinstance(exc.stdout, bytes)
                    else exc.stdout
                )
            ),
            stderr=(
                ""
                if exc.stderr is None
                else (
                    exc.stderr.decode("utf-8", errors="replace")
                    if isinstance(exc.stderr, bytes)
                    else exc.stderr
                )
            ),
        )
    return RawProbeExecution(
        command_role=role,
        returncode=result.returncode,
        timed_out=False,
        duration_ms=int((time.monotonic() - started) * 1000),
        started_at=started_at,
        stdout=result.stdout,
        stderr=result.stderr,
    )


def evaluate_semantics(
    raw: RawProbeExecution,
    parser: Callable[[RawProbeExecution], ObservationT],
    validator: Callable[[ObservationT], ProbeDecision],
) -> tuple[ObservationT | None, ProbeDecision]:
    if raw.timed_out:
        return None, ProbeDecision(
            status=ProbeStatus.FAILED,
            failure_code=FailureCode.SUBPROCESS_TIMEOUT,
            detail="probe subprocess timed out",
        )
    if raw.returncode != 0:
        return None, ProbeDecision(
            status=ProbeStatus.FAILED,
            failure_code=FailureCode.SUBPROCESS_FAILED,
            detail="probe subprocess returned nonzero",
        )
    try:
        observation = parser(raw)
    except Exception as exc:
        return None, ProbeDecision(
            status=ProbeStatus.FAILED,
            failure_code=FailureCode.SEMANTIC_PARSE_FAILED,
            detail=f"semantic parser failed: {type(exc).__name__}",
        )
    try:
        decision = validator(observation)
    except Exception as exc:
        return observation, ProbeDecision(
            status=ProbeStatus.FAILED,
            failure_code=FailureCode.SEMANTIC_CONTRACT_FAILED,
            detail=f"semantic validator failed: {type(exc).__name__}",
        )
    return observation, decision


def run_semantic_probe(
    role: str,
    argv: list[str],
    parser: Callable[[RawProbeExecution], ObservationT],
    validator: Callable[[ObservationT], ProbeDecision],
    *,
    timeout: float,
    dependencies: tuple[str, ...],
    environment: dict[str, str] | None = None,
) -> ProbeOutcome[ObservationT]:
    raw = run_probe_raw(
        role,
        argv,
        timeout=timeout,
        environment=environment,
    )
    observation, decision = evaluate_semantics(
        raw,
        parser,
        validator,
    )
    evidence = project_evidence(
        raw,
        decision,
        dependencies=dependencies,
    )
    return ProbeOutcome(
        observation=observation,
        decision=decision,
        evidence=evidence,
    )


def run_command_probe(
    role: str,
    argv: list[str],
    *,
    timeout: float,
    dependencies: tuple[str, ...],
    environment: dict[str, str] | None = None,
) -> ProbeOutcome[None]:
    raw = run_probe_raw(
        role,
        argv,
        timeout=timeout,
        environment=environment,
    )
    if raw.timed_out:
        decision = ProbeDecision(
            status=ProbeStatus.FAILED,
            failure_code=FailureCode.SUBPROCESS_TIMEOUT,
            detail="probe subprocess timed out",
        )
    elif raw.returncode != 0:
        decision = ProbeDecision(
            status=ProbeStatus.FAILED,
            failure_code=FailureCode.SUBPROCESS_FAILED,
            detail="probe subprocess returned nonzero",
        )
    else:
        decision = ProbeDecision(status=ProbeStatus.PASSED)
    evidence = project_evidence(
        raw,
        decision,
        dependencies=dependencies,
    )
    return ProbeOutcome(
        observation=None,
        decision=decision,
        evidence=evidence,
    )


def blocked_outcome(
    role: str,
    dependencies: tuple[str, ...],
    outcomes: dict[str, ProbeOutcome[object]],
) -> ProbeOutcome[None]:
    failed = [
        name
        for name in dependencies
        if outcomes[name].decision.status != ProbeStatus.PASSED
    ]
    decision = ProbeDecision(
        status=ProbeStatus.BLOCKED_BY_UPSTREAM_FAILURE,
        failure_code=FailureCode.BLOCKED_BY_UPSTREAM_FAILURE,
        detail="blocked_by=" + ",".join(failed),
    )
    evidence = project_local_evidence(
        role,
        decision,
        dependencies=dependencies,
        started_at=None,
        duration_ms=0,
    )
    return ProbeOutcome(
        observation=None,
        decision=decision,
        evidence=evidence,
    )


def dependencies_passed(
    outcomes: dict[str, ProbeOutcome[object]],
    dependencies: tuple[str, ...],
) -> bool:
    return all(
        outcomes[name].decision.status == ProbeStatus.PASSED
        for name in dependencies
    )


def _load_json_object(text: str) -> dict[str, object]:
    payload = json.loads(text)
    if not isinstance(payload, dict):
        raise ValueError("expected one JSON object")
    return payload


def _require_string(
    payload: dict[str, object],
    key: str,
) -> str:
    value = payload.get(key)
    if not isinstance(value, str):
        raise ValueError(f"{key} is not a string")
    return value


def _require_bool(
    payload: dict[str, object],
    key: str,
) -> bool:
    value = payload.get(key)
    if not isinstance(value, bool):
        raise ValueError(f"{key} is not a boolean")
    return value


def _require_int(
    payload: dict[str, object],
    key: str,
) -> int:
    value = payload.get(key)
    if isinstance(value, bool) or not isinstance(value, int):
        raise ValueError(f"{key} is not an integer")
    return value


def parse_python_version(
    raw: RawProbeExecution,
) -> PythonVersionObservation:
    payload = _load_json_object(raw.stdout.strip())
    return PythonVersionObservation(
        python=_require_string(payload, "python")
    )


def validate_python_version(
    observation: PythonVersionObservation,
) -> ProbeDecision:
    if not observation.python.startswith(
        EXPECTED_RUNTIME["python_major_minor"] + "."
    ):
        return ProbeDecision(
            status=ProbeStatus.FAILED,
            failure_code=FailureCode.SEMANTIC_CONTRACT_FAILED,
            detail="Python major/minor identity drifted",
        )
    return ProbeDecision(status=ProbeStatus.PASSED)


def parse_distribution_snapshot(
    raw: RawProbeExecution,
) -> DistributionSnapshotObservation:
    payload = _load_json_object(raw.stdout.strip())
    sha256 = _require_string(payload, "sha256")
    if _SHA256_PATTERN.fullmatch(sha256) is None:
        raise ValueError("distribution snapshot SHA-256 is invalid")
    return DistributionSnapshotObservation(
        count=_require_int(payload, "count"),
        sha256=sha256,
    )


def validate_distribution_snapshot(
    observation: DistributionSnapshotObservation,
) -> ProbeDecision:
    if observation.count < 1:
        return ProbeDecision(
            status=ProbeStatus.FAILED,
            failure_code=FailureCode.SEMANTIC_CONTRACT_FAILED,
            detail="distribution snapshot is unexpectedly empty",
        )
    return ProbeDecision(status=ProbeStatus.PASSED)


def parse_gpu_topology(
    raw: RawProbeExecution,
) -> GpuTopologyObservation:
    rows = tuple(
        row.strip()
        for row in raw.stdout.splitlines()
        if row.strip()
    )
    return GpuTopologyObservation(rows=rows)


def validate_gpu_topology(
    observation: GpuTopologyObservation,
) -> ProbeDecision:
    if len(observation.rows) != 2:
        return ProbeDecision(
            status=ProbeStatus.FAILED,
            failure_code=FailureCode.SEMANTIC_CONTRACT_FAILED,
            detail="expected exactly two GPUs",
        )
    if any("T4" not in row for row in observation.rows):
        return ProbeDecision(
            status=ProbeStatus.FAILED,
            failure_code=FailureCode.SEMANTIC_CONTRACT_FAILED,
            detail="expected both GPUs to be T4",
        )
    return ProbeDecision(status=ProbeStatus.PASSED)


def parse_target_identity(
    raw: RawProbeExecution,
) -> TargetIdentityObservation:
    payload = _load_json_object(raw.stdout.strip())
    return TargetIdentityObservation(
        python=_require_string(payload, "python"),
        prefix_matches_expected=_require_bool(
            payload,
            "prefix_matches_expected",
        ),
        base_prefix_differs=_require_bool(
            payload,
            "base_prefix_differs",
        ),
        user_site_enabled=_require_bool(
            payload,
            "user_site_enabled",
        ),
        purelib_within_prefix=_require_bool(
            payload,
            "purelib_within_prefix",
        ),
        platlib_within_prefix=_require_bool(
            payload,
            "platlib_within_prefix",
        ),
        system_site_packages_enabled=_require_bool(
            payload,
            "system_site_packages_enabled",
        ),
        pip_present=_require_bool(payload, "pip_present"),
    )


def validate_target_identity(
    observation: TargetIdentityObservation,
) -> ProbeDecision:
    required = (
        observation.prefix_matches_expected is True,
        observation.base_prefix_differs is True,
        observation.user_site_enabled is False,
        observation.purelib_within_prefix is True,
        observation.platlib_within_prefix is True,
        observation.system_site_packages_enabled is False,
        observation.pip_present is False,
        observation.python.startswith(
            EXPECTED_RUNTIME["python_major_minor"] + "."
        ),
    )
    if not all(required):
        return ProbeDecision(
            status=ProbeStatus.FAILED,
            failure_code=FailureCode.SEMANTIC_CONTRACT_FAILED,
            detail="target interpreter isolation contract drifted",
        )
    return ProbeDecision(status=ProbeStatus.PASSED)


def parse_distribution_inventory(
    raw: RawProbeExecution,
) -> DistributionInventoryObservation:
    payload = json.loads(raw.stdout.strip())
    if not isinstance(payload, list):
        raise ValueError("target distribution inventory is not a list")
    items: list[tuple[str, str]] = []
    for row in payload:
        if (
            not isinstance(row, list)
            or len(row) != 2
            or not all(isinstance(value, str) for value in row)
        ):
            raise ValueError("target distribution row is invalid")
        items.append((row[0], row[1]))
    return DistributionInventoryObservation(items=tuple(items))


def validate_distribution_inventory(
    observation: DistributionInventoryObservation,
    *,
    expected_items: tuple[tuple[str, str], ...],
) -> ProbeDecision:
    if observation.items != expected_items:
        return ProbeDecision(
            status=ProbeStatus.FAILED,
            failure_code=FailureCode.SEMANTIC_CONTRACT_FAILED,
            detail="target distribution inventory differs from exact lock",
        )
    return ProbeDecision(status=ProbeStatus.PASSED)


def parse_controlled_startup(
    raw: RawProbeExecution,
) -> ControlledStartupObservation:
    payload = _load_json_object(raw.stdout.strip())
    external = payload.get("external_package_paths")
    if not isinstance(external, list):
        raise ValueError("external_package_paths is invalid")
    if not all(isinstance(item, str) for item in external):
        raise ValueError("external_package_paths contains non-string")
    return ControlledStartupObservation(
        prefix=Path(_require_string(payload, "prefix")),
        base_prefix=Path(_require_string(payload, "base_prefix")),
        no_site_flag=_require_int(payload, "no_site_flag"),
        user_site_enabled=_require_bool(
            payload,
            "user_site_enabled",
        ),
        target_site_present=_require_bool(
            payload,
            "target_site_present",
        ),
        external_package_paths=tuple(
            Path(item) for item in external
        ),
        sitecustomize_file=_require_string(
            payload,
            "sitecustomize_file",
        ),
        usercustomize_file=_require_string(
            payload,
            "usercustomize_file",
        ),
        pythonpath_present=_require_bool(
            payload,
            "pythonpath_present",
        ),
        pythonhome_present=_require_bool(
            payload,
            "pythonhome_present",
        ),
        ld_preload_present=_require_bool(
            payload,
            "ld_preload_present",
        ),
        python_no_user_site=_require_string(
            payload,
            "python_no_user_site",
        ),
    )


def validate_controlled_startup(
    observation: ControlledStartupObservation,
    *,
    expected_root: Path,
) -> ProbeDecision:
    expected = expected_root.resolve()
    if observation.prefix.resolve() != expected:
        return ProbeDecision(
            status=ProbeStatus.FAILED,
            failure_code=FailureCode.SEMANTIC_CONTRACT_FAILED,
            detail="controlled startup target prefix drifted",
        )
    if observation.base_prefix.resolve() == observation.prefix.resolve():
        return ProbeDecision(
            status=ProbeStatus.FAILED,
            failure_code=FailureCode.SEMANTIC_CONTRACT_FAILED,
            detail="base_prefix unexpectedly equals target prefix",
        )
    required = (
        observation.no_site_flag == 1,
        observation.user_site_enabled is False,
        observation.target_site_present is True,
        not observation.external_package_paths,
        observation.sitecustomize_file
        == "<auragateway-suppressed-sitecustomize>",
        observation.usercustomize_file
        == "<auragateway-suppressed-usercustomize>",
        observation.pythonpath_present is False,
        observation.pythonhome_present is False,
        observation.ld_preload_present is False,
        observation.python_no_user_site == "1",
    )
    if not all(required):
        return ProbeDecision(
            status=ProbeStatus.FAILED,
            failure_code=FailureCode.SEMANTIC_CONTRACT_FAILED,
            detail="controlled startup isolation contract drifted",
        )
    return ProbeDecision(status=ProbeStatus.PASSED)


def parse_native_inventory(
    raw: RawProbeExecution,
) -> NativeInventoryObservation:
    payload = _load_json_object(raw.stdout.strip())
    required_raw = payload.get("required")
    legacy_raw = payload.get("legacy_c_candidates", [])
    optional_raw = payload.get("optional_candidates", [])
    if not isinstance(required_raw, list):
        raise ValueError("required native inventory is invalid")
    if not isinstance(legacy_raw, list):
        raise ValueError("legacy native inventory is invalid")
    if not isinstance(optional_raw, list):
        raise ValueError("optional native inventory is invalid")

    required: list[NativeFileObservation] = []
    for item in required_raw:
        if not isinstance(item, dict):
            raise ValueError("native inventory entry is invalid")
        path_value = item.get("path")
        sha_value = item.get("sha256")
        size_value = item.get("size_bytes")
        if not isinstance(path_value, str):
            raise ValueError("native inventory path is invalid")
        if (
            not isinstance(sha_value, str)
            or _SHA256_PATTERN.fullmatch(sha_value) is None
        ):
            raise ValueError("native inventory SHA-256 is invalid")
        if (
            isinstance(size_value, bool)
            or not isinstance(size_value, int)
            or size_value < 0
        ):
            raise ValueError("native inventory size is invalid")
        required.append(
            NativeFileObservation(
                path=Path(path_value),
                sha256=sha_value,
                size_bytes=size_value,
            )
        )

    if not all(isinstance(item, str) for item in legacy_raw):
        raise ValueError("legacy native path is invalid")
    if not all(isinstance(item, str) for item in optional_raw):
        raise ValueError("optional native path is invalid")

    return NativeInventoryObservation(
        required=tuple(required),
        legacy_c_candidates=tuple(
            Path(item) for item in legacy_raw
        ),
        optional_candidates=tuple(
            Path(item) for item in optional_raw
        ),
    )


def _canonical_existing_path(path: Path) -> Path | None:
    try:
        return path.resolve(strict=True)
    except (FileNotFoundError, OSError, RuntimeError):
        return None


def _is_within_resolved(path: Path, root: Path) -> bool:
    try:
        path.relative_to(root)
    except ValueError:
        return False
    return True


def validate_native_inventory(
    observation: NativeInventoryObservation,
    *,
    target_vllm_root: Path,
) -> ProbeDecision:
    if len(observation.required) != 1:
        return ProbeDecision(
            status=ProbeStatus.FAILED,
            failure_code=FailureCode.SEMANTIC_CONTRACT_FAILED,
            detail="expected exactly one required native extension",
        )
    native_path = _canonical_existing_path(
        observation.required[0].path
    )
    vllm_root = _canonical_existing_path(target_vllm_root)
    if native_path is None or vllm_root is None:
        return ProbeDecision(
            status=ProbeStatus.FAILED,
            failure_code=FailureCode.UNKNOWN_NATIVE_ORIGIN,
            detail="required native extension path unavailable",
        )
    if not _is_within_resolved(native_path, vllm_root):
        return ProbeDecision(
            status=ProbeStatus.FAILED,
            failure_code=FailureCode.UNKNOWN_NATIVE_ORIGIN,
            detail="required native extension escaped target vllm root",
        )
    if not native_path.name.startswith("_C_stable_libtorch"):
        return ProbeDecision(
            status=ProbeStatus.FAILED,
            failure_code=FailureCode.SEMANTIC_CONTRACT_FAILED,
            detail="required native extension filename drifted",
        )
    return ProbeDecision(status=ProbeStatus.PASSED)


def _path_is_within(path: Path, root: Path) -> bool:
    try:
        path.resolve().relative_to(root.resolve())
    except ValueError:
        return False
    return True


def _is_prohibited_library_path(value: str) -> bool:
    normalized = value.replace("\\", "/").rstrip("/")
    return any(
        marker in normalized
        for marker in PROHIBITED_LIBRARY_PATH_MARKERS
    )


def _is_external_python_package_path(
    value: str,
    target_site: Path,
) -> bool:
    path = Path(value)
    parts = set(path.parts)
    return (
        bool({"site-packages", "dist-packages"} & parts)
        and not _path_is_within(path, target_site)
    )


def target_site_path() -> Path:
    return (
        TARGET_ROOT
        / "lib"
        / f"python{sys.version_info.major}.{sys.version_info.minor}"
        / "site-packages"
    )


def target_nvidia_library_directories(
    target_site: Path,
) -> tuple[Path, ...]:
    return tuple(
        sorted(
            path
            for path in target_site.glob("nvidia/*/lib")
            if path.is_dir()
        )
    )


def target_torch_library_directory(
    target_site: Path,
) -> Path:
    return target_site / "torch" / "lib"


def build_canonical_loader_environment(
    target_site: Path,
) -> tuple[
    dict[str, str],
    CanonicalLoaderObservation,
    ProbeDecision,
]:
    target_site = target_site.resolve()
    target_nvidia = target_nvidia_library_directories(
        target_site
    )
    target_torch = target_torch_library_directory(
        target_site
    )
    if not target_nvidia:
        return {}, CanonicalLoaderObservation(
            target_nvidia_library_count=0,
            target_torch_library=target_torch,
            real_driver_directory=REAL_DRIVER_DIRECTORY,
            ld_preload_absent=False,
            pythonpath_absent=False,
            pythonhome_absent=False,
            prohibited_path_present=False,
        ), ProbeDecision(
            status=ProbeStatus.FAILED,
            failure_code=FailureCode.SEMANTIC_CONTRACT_FAILED,
            detail="target NVIDIA library directories unavailable",
        )
    if not target_torch.is_dir():
        return {}, CanonicalLoaderObservation(
            target_nvidia_library_count=len(target_nvidia),
            target_torch_library=target_torch,
            real_driver_directory=REAL_DRIVER_DIRECTORY,
            ld_preload_absent=False,
            pythonpath_absent=False,
            pythonhome_absent=False,
            prohibited_path_present=False,
        ), ProbeDecision(
            status=ProbeStatus.FAILED,
            failure_code=FailureCode.SEMANTIC_CONTRACT_FAILED,
            detail="target Torch library directory unavailable",
        )
    if not REAL_DRIVER_DIRECTORY.is_dir():
        return {}, CanonicalLoaderObservation(
            target_nvidia_library_count=len(target_nvidia),
            target_torch_library=target_torch,
            real_driver_directory=REAL_DRIVER_DIRECTORY,
            ld_preload_absent=False,
            pythonpath_absent=False,
            pythonhome_absent=False,
            prohibited_path_present=False,
        ), ProbeDecision(
            status=ProbeStatus.FAILED,
            failure_code=FailureCode.SEMANTIC_CONTRACT_FAILED,
            detail="real NVIDIA driver directory unavailable",
        )

    environment = dict(os.environ)
    environment.pop("PYTHONPATH", None)
    environment.pop("PYTHONHOME", None)
    environment.pop("LD_PRELOAD", None)

    inherited: list[str] = []
    for value in environment.get(
        "LD_LIBRARY_PATH",
        "",
    ).split(os.pathsep):
        if not value:
            continue
        if _is_prohibited_library_path(value):
            continue
        if _is_external_python_package_path(
            value,
            target_site,
        ):
            continue
        inherited.append(value)

    ordered: list[str] = []
    for value in (
        *(
            str(path.resolve())
            for path in target_nvidia
        ),
        str(target_torch.resolve()),
        str(REAL_DRIVER_DIRECTORY.resolve()),
        *inherited,
    ):
        if value not in ordered:
            ordered.append(value)

    prohibited = any(
        _is_prohibited_library_path(value)
        for value in ordered
    )
    external = any(
        _is_external_python_package_path(
            value,
            target_site,
        )
        for value in ordered
    )
    if prohibited or external:
        decision = ProbeDecision(
            status=ProbeStatus.FAILED,
            failure_code=FailureCode.PROHIBITED_NATIVE_ORIGIN,
            detail="canonical loader retained prohibited path",
        )
    else:
        decision = ProbeDecision(status=ProbeStatus.PASSED)

    environment.update(
        {
            "PIP_DISABLE_PIP_VERSION_CHECK": "1",
            "PIP_NO_INDEX": "1",
            "PIP_NO_CACHE_DIR": "1",
            "HF_HUB_OFFLINE": "1",
            "TRANSFORMERS_OFFLINE": "1",
            "PYTHONNOUSERSITE": "1",
            "VIRTUAL_ENV": str(TARGET_ROOT.resolve()),
            "LD_LIBRARY_PATH": os.pathsep.join(ordered),
        }
    )
    inherited_path = environment.get("PATH", "")
    environment["PATH"] = os.pathsep.join(
        value
        for value in (
            str((TARGET_ROOT / "bin").resolve()),
            inherited_path,
        )
        if value
    )
    observation = CanonicalLoaderObservation(
        target_nvidia_library_count=len(target_nvidia),
        target_torch_library=target_torch.resolve(),
        real_driver_directory=REAL_DRIVER_DIRECTORY.resolve(),
        ld_preload_absent="LD_PRELOAD" not in environment,
        pythonpath_absent="PYTHONPATH" not in environment,
        pythonhome_absent="PYTHONHOME" not in environment,
        prohibited_path_present=prohibited,
    )
    return environment, observation, decision


def is_governed_native_path(path: Path) -> bool:
    lexical = path.as_posix()
    if any(
        marker in lexical
        for marker in PROHIBITED_LIBRARY_PATH_MARKERS
    ):
        return True
    if {"site-packages", "dist-packages"} & set(path.parts):
        return True
    return path.name.startswith(
        GOVERNED_NATIVE_LIBRARY_BASENAMES
    )


def classify_native_origin(
    path: Path,
    *,
    target_site: Path,
    real_driver_root: Path,
) -> NativeOriginClass:
    lexical = path.as_posix()
    if any(
        marker in lexical
        for marker in PROHIBITED_LIBRARY_PATH_MARKERS
    ):
        return NativeOriginClass.PROHIBITED_AMBIENT

    resolved = _canonical_existing_path(path)
    target_resolved = _canonical_existing_path(target_site)
    driver_resolved = _canonical_existing_path(
        real_driver_root
    )
    if resolved is None or target_resolved is None:
        return NativeOriginClass.UNKNOWN
    if _is_within_resolved(resolved, target_resolved):
        return NativeOriginClass.TARGET_OWNED
    if (
        driver_resolved is not None
        and _is_within_resolved(
            resolved,
            driver_resolved,
        )
    ):
        return NativeOriginClass.PERMITTED_HOST_PLATFORM
    if {"site-packages", "dist-packages"} & set(
        resolved.parts
    ):
        return NativeOriginClass.PROHIBITED_AMBIENT
    return NativeOriginClass.UNKNOWN


def validate_native_origin_set(
    paths: tuple[Path, ...],
    *,
    target_site: Path,
    real_driver_root: Path,
    permitted: frozenset[NativeOriginClass],
) -> ProbeDecision:
    governed = tuple(
        path for path in paths
        if is_governed_native_path(path)
    )
    for path in governed:
        origin = classify_native_origin(
            path,
            target_site=target_site,
            real_driver_root=real_driver_root,
        )
        if origin == NativeOriginClass.PROHIBITED_AMBIENT:
            return ProbeDecision(
                status=ProbeStatus.FAILED,
                failure_code=FailureCode.PROHIBITED_NATIVE_ORIGIN,
                detail="prohibited ambient native origin observed",
            )
        if origin == NativeOriginClass.UNKNOWN:
            return ProbeDecision(
                status=ProbeStatus.FAILED,
                failure_code=FailureCode.UNKNOWN_NATIVE_ORIGIN,
                detail="unknown native origin observed",
            )
        if origin not in permitted:
            return ProbeDecision(
                status=ProbeStatus.FAILED,
                failure_code=FailureCode.SEMANTIC_CONTRACT_FAILED,
                detail="native origin class is not permitted",
            )
    return ProbeDecision(status=ProbeStatus.PASSED)


def parse_torch_family(
    raw: RawProbeExecution,
) -> TorchFamilyObservation:
    payload = _load_json_object(raw.stdout.strip())
    names = payload.get("cuda_device_names")
    if not isinstance(names, list):
        raise ValueError("CUDA device names invalid")
    if not all(isinstance(name, str) for name in names):
        raise ValueError("CUDA device name is not a string")
    return TorchFamilyObservation(
        torch=_require_string(payload, "torch"),
        torchaudio=_require_string(payload, "torchaudio"),
        torchvision=_require_string(payload, "torchvision"),
        torch_cuda_version=_require_string(
            payload,
            "torch_cuda_version",
        ),
        cuda_available=_require_bool(
            payload,
            "cuda_available",
        ),
        cuda_device_count=_require_int(
            payload,
            "cuda_device_count",
        ),
        cuda_device_names=tuple(names),
    )


def validate_torch_family(
    observation: TorchFamilyObservation,
) -> ProbeDecision:
    expected = (
        observation.torch == EXPECTED_RUNTIME["torch"],
        observation.torchaudio
        == EXPECTED_RUNTIME["torchaudio"],
        observation.torchvision
        == EXPECTED_RUNTIME["torchvision"],
        observation.torch_cuda_version
        == EXPECTED_RUNTIME["torch_cuda_version"],
        observation.cuda_available is True,
        observation.cuda_device_count == 2,
        len(observation.cuda_device_names) == 2,
        all(
            "T4" in name
            for name in observation.cuda_device_names
        ),
    )
    if not all(expected):
        return ProbeDecision(
            status=ProbeStatus.FAILED,
            failure_code=FailureCode.SEMANTIC_CONTRACT_FAILED,
            detail="torch-family runtime contract drifted",
        )
    return ProbeDecision(status=ProbeStatus.PASSED)


def parse_version_field(
    raw: RawProbeExecution,
    *,
    key: str,
    package: str,
) -> VersionObservation:
    payload = _load_json_object(raw.stdout.strip())
    return VersionObservation(
        package=package,
        version=_require_string(payload, key),
    )


def validate_exact_version(
    observation: VersionObservation,
    *,
    expected: str,
) -> ProbeDecision:
    if observation.version != expected:
        return ProbeDecision(
            status=ProbeStatus.FAILED,
            failure_code=FailureCode.SEMANTIC_CONTRACT_FAILED,
            detail=f"{observation.package} version drifted",
        )
    return ProbeDecision(status=ProbeStatus.PASSED)


def parse_ldd(
    raw: RawProbeExecution,
) -> NativeLinkerObservation:
    unresolved = "not found" in raw.stdout.lower()
    resolved: list[Path] = []
    for line in raw.stdout.splitlines():
        if "=>" not in line:
            continue
        right = line.split("=>", 1)[1].strip()
        candidate = right.split(" ", 1)[0]
        candidate_path = Path(candidate)
        if candidate_path.is_absolute():
            resolved.append(candidate_path)
    return NativeLinkerObservation(
        unresolved_required_library=unresolved,
        resolved_paths=tuple(resolved),
    )


def validate_native_linker(
    observation: NativeLinkerObservation,
    *,
    target_site: Path,
) -> ProbeDecision:
    if observation.unresolved_required_library:
        return ProbeDecision(
            status=ProbeStatus.FAILED,
            failure_code=FailureCode.SEMANTIC_CONTRACT_FAILED,
            detail="static linker reported unresolved library",
        )
    origin_decision = validate_native_origin_set(
        observation.resolved_paths,
        target_site=target_site,
        real_driver_root=REAL_DRIVER_DIRECTORY,
        permitted=frozenset(
            {
                NativeOriginClass.TARGET_OWNED,
                NativeOriginClass.PERMITTED_HOST_PLATFORM,
            }
        ),
    )
    if origin_decision.status != ProbeStatus.PASSED:
        return origin_decision

    governed = tuple(
        path
        for path in observation.resolved_paths
        if is_governed_native_path(path)
    )
    torch_paths = tuple(
        path
        for path in governed
        if path.name.startswith(("libtorch", "libc10"))
    )
    nvidia_paths = tuple(
        path
        for path in governed
        if path.name.startswith(
            (
                "libcudart",
                "libnvJitLink",
                "libcusparse",
                "libcublas",
                "libcufft",
                "libcurand",
                "libcusolver",
                "libnccl",
                "libnvrtc",
            )
        )
    )
    if not torch_paths:
        return ProbeDecision(
            status=ProbeStatus.FAILED,
            failure_code=FailureCode.SEMANTIC_CONTRACT_FAILED,
            detail="static linker did not resolve target Torch libraries",
        )
    if not nvidia_paths:
        return ProbeDecision(
            status=ProbeStatus.FAILED,
            failure_code=FailureCode.SEMANTIC_CONTRACT_FAILED,
            detail="static linker did not resolve target NVIDIA libraries",
        )
    return ProbeDecision(status=ProbeStatus.PASSED)


def parse_native_extension(
    raw: RawProbeExecution,
) -> NativeExtensionObservation:
    payload = _load_json_object(raw.stdout.strip())
    return NativeExtensionObservation(
        native_extension=_require_string(
            payload,
            "native_extension",
        ),
        file=Path(_require_string(payload, "file")),
    )


def validate_native_extension(
    observation: NativeExtensionObservation,
    *,
    target_vllm_root: Path,
) -> ProbeDecision:
    if observation.native_extension != REQUIRED_NATIVE_MODULE:
        return ProbeDecision(
            status=ProbeStatus.FAILED,
            failure_code=FailureCode.SEMANTIC_CONTRACT_FAILED,
            detail="required native module identity drifted",
        )
    resolved = _canonical_existing_path(observation.file)
    root = _canonical_existing_path(target_vllm_root)
    if (
        resolved is None
        or root is None
        or not _is_within_resolved(resolved, root)
    ):
        return ProbeDecision(
            status=ProbeStatus.FAILED,
            failure_code=FailureCode.UNKNOWN_NATIVE_ORIGIN,
            detail="native module imported outside target vllm root",
        )
    return ProbeDecision(status=ProbeStatus.PASSED)


def parse_native_runtime_provenance(
    raw: RawProbeExecution,
) -> NativeRuntimeProvenanceObservation:
    payload = _load_json_object(raw.stdout.strip())
    loaded = payload.get("loaded_paths")
    if not isinstance(loaded, list):
        raise ValueError("loaded_paths is invalid")
    if not all(isinstance(item, str) for item in loaded):
        raise ValueError("loaded_paths contains non-string")
    return NativeRuntimeProvenanceObservation(
        native_module=_require_string(
            payload,
            "native_module",
        ),
        native_file=Path(
            _require_string(payload, "native_file")
        ),
        torch_file=Path(
            _require_string(payload, "torch_file")
        ),
        vllm_file=Path(
            _require_string(payload, "vllm_file")
        ),
        cuda_available=_require_bool(
            payload,
            "cuda_available",
        ),
        loaded_paths=tuple(Path(item) for item in loaded),
    )


def validate_native_runtime_provenance(
    observation: NativeRuntimeProvenanceObservation,
    *,
    target_site: Path,
) -> ProbeDecision:
    target_site = target_site.resolve()
    if observation.native_module != REQUIRED_NATIVE_MODULE:
        return ProbeDecision(
            status=ProbeStatus.FAILED,
            failure_code=FailureCode.SEMANTIC_CONTRACT_FAILED,
            detail="native runtime module identity drifted",
        )
    native_path = _canonical_existing_path(
        observation.native_file
    )
    torch_path = _canonical_existing_path(
        observation.torch_file
    )
    vllm_path = _canonical_existing_path(
        observation.vllm_file
    )
    if (
        native_path is None
        or torch_path is None
        or vllm_path is None
    ):
        return ProbeDecision(
            status=ProbeStatus.FAILED,
            failure_code=FailureCode.UNKNOWN_NATIVE_ORIGIN,
            detail="runtime Python package origin unavailable",
        )
    if not _is_within_resolved(
        native_path,
        (target_site / "vllm").resolve(),
    ):
        return ProbeDecision(
            status=ProbeStatus.FAILED,
            failure_code=FailureCode.UNKNOWN_NATIVE_ORIGIN,
            detail="native module runtime origin drifted",
        )
    if not _is_within_resolved(
        torch_path,
        (target_site / "torch").resolve(),
    ):
        return ProbeDecision(
            status=ProbeStatus.FAILED,
            failure_code=FailureCode.UNKNOWN_NATIVE_ORIGIN,
            detail="torch Python runtime origin drifted",
        )
    if not _is_within_resolved(
        vllm_path,
        (target_site / "vllm").resolve(),
    ):
        return ProbeDecision(
            status=ProbeStatus.FAILED,
            failure_code=FailureCode.UNKNOWN_NATIVE_ORIGIN,
            detail="vLLM Python runtime origin drifted",
        )
    if observation.cuda_available is not True:
        return ProbeDecision(
            status=ProbeStatus.FAILED,
            failure_code=FailureCode.SEMANTIC_CONTRACT_FAILED,
            detail="CUDA unavailable during native provenance probe",
        )
    origin_decision = validate_native_origin_set(
        observation.loaded_paths,
        target_site=target_site,
        real_driver_root=REAL_DRIVER_DIRECTORY,
        permitted=frozenset(
            {
                NativeOriginClass.TARGET_OWNED,
                NativeOriginClass.PERMITTED_HOST_PLATFORM,
            }
        ),
    )
    if origin_decision.status != ProbeStatus.PASSED:
        return origin_decision

    governed = tuple(
        path
        for path in observation.loaded_paths
        if is_governed_native_path(path)
    )
    torch_loaded = tuple(
        path
        for path in governed
        if path.name.startswith(("libtorch", "libc10"))
    )
    nvidia_loaded = tuple(
        path
        for path in governed
        if path.name.startswith(
            (
                "libcudart",
                "libnvJitLink",
                "libcusparse",
                "libcublas",
                "libcufft",
                "libcurand",
                "libcusolver",
                "libnccl",
                "libnvrtc",
            )
        )
    )
    driver_loaded = tuple(
        path
        for path in governed
        if path.name.startswith("libcuda.so")
    )
    if not torch_loaded:
        return ProbeDecision(
            status=ProbeStatus.FAILED,
            failure_code=FailureCode.SEMANTIC_CONTRACT_FAILED,
            detail="no governed target Torch native library observed",
        )
    if not nvidia_loaded:
        return ProbeDecision(
            status=ProbeStatus.FAILED,
            failure_code=FailureCode.SEMANTIC_CONTRACT_FAILED,
            detail="no governed target NVIDIA runtime library observed",
        )
    if not driver_loaded:
        return ProbeDecision(
            status=ProbeStatus.FAILED,
            failure_code=FailureCode.SEMANTIC_CONTRACT_FAILED,
            detail="real NVIDIA driver library was not observed",
        )
    return ProbeDecision(status=ProbeStatus.PASSED)


def parse_cuda_platform(
    raw: RawProbeExecution,
) -> CudaPlatformObservation:
    payload = _load_json_object(raw.stdout.strip())
    names = payload.get("cuda_device_names")
    if not isinstance(names, list):
        raise ValueError("CUDA platform names invalid")
    if not all(isinstance(name, str) for name in names):
        raise ValueError("CUDA platform name is non-string")
    return CudaPlatformObservation(
        cuda_available=_require_bool(
            payload,
            "cuda_available",
        ),
        cuda_device_count=_require_int(
            payload,
            "cuda_device_count",
        ),
        cuda_device_names=tuple(names),
        required_native_module_loaded=_require_bool(
            payload,
            "required_native_module_loaded",
        ),
    )


def validate_cuda_platform(
    observation: CudaPlatformObservation,
) -> ProbeDecision:
    valid = (
        observation.cuda_available is True
        and observation.cuda_device_count == 2
        and len(observation.cuda_device_names) == 2
        and all(
            "T4" in name
            for name in observation.cuda_device_names
        )
        and observation.required_native_module_loaded is True
    )
    if not valid:
        return ProbeDecision(
            status=ProbeStatus.FAILED,
            failure_code=FailureCode.SEMANTIC_CONTRACT_FAILED,
            detail="vLLM CUDA platform capability drifted",
        )
    return ProbeDecision(status=ProbeStatus.PASSED)


def validate_safe_relative_path(value: str) -> PurePosixPath:
    path = PurePosixPath(value)
    if path.is_absolute():
        raise ValueError("absolute manifest path prohibited")
    if any(
        part in {"", ".", ".."}
        for part in path.parts
    ):
        raise ValueError("unsafe manifest path")
    return path


def load_object(path: Path) -> dict[str, object]:
    payload = json.loads(path.read_text(encoding="utf-8"))
    if not isinstance(payload, dict):
        raise ValueError("expected JSON object")
    return payload


def expected_requirements(
    records_: tuple[dict[str, object], ...],
) -> str:
    rows: list[str] = []
    for record in sorted(
        records_,
        key=lambda item: str(item["normalized_name"]),
    ):
        rows.append(
            f'{record["normalized_name"]}=={record["version"]} '
            f'--hash=sha256:{record["sha256"]}'
        )
    return "\n".join(rows) + "\n"


def expected_materialization_lock(
    records_: tuple[dict[str, object], ...],
) -> str:
    rows: list[str] = []
    for record in sorted(
        records_,
        key=lambda item: str(item["normalized_name"]),
    ):
        rows.append(
            f'{record["sha256"]}  '
            f'wheels/{record["artifact_filename"]}'
        )
    return "\n".join(rows) + "\n"


def discover_and_validate_input() -> ProbeOutcome[
    InputValidationObservation
]:
    role = "input_validation"
    started = time.monotonic()
    started_at = datetime.now(UTC).isoformat(
        timespec="seconds"
    )
    diagnostic_text = ""
    observation: InputValidationObservation | None = None

    try:
        matches = tuple(
            path
            for path in Path("/kaggle/input").rglob(
                INPUT_DIRECTORY_NAME
            )
            if path.is_dir()
        )
        if len(matches) != 1:
            raise ValueError(
                "expected exactly one accepted materializer output "
                f"directory; observed={len(matches)}"
            )
        input_root = matches[0]

        observed_top = frozenset(
            path.name for path in input_root.iterdir()
        )
        if observed_top != EXPECTED_TOP_LEVEL:
            raise ValueError(
                "wheelhouse top-level topology drifted"
            )

        symlinks = tuple(
            path
            for path in input_root.rglob("*")
            if path.is_symlink()
        )
        if symlinks:
            raise ValueError(
                "wheelhouse contains prohibited symlinks"
            )

        for name, expected_sha in (
            EXPECTED_CONTROL_HASHES.items()
        ):
            if streaming_sha256(
                input_root / name
            ) != expected_sha:
                raise ValueError(
                    f"control artifact SHA drifted: {name}"
                )

        lock = load_object(
            input_root / "resolution_lock.json"
        )
        if lock.get("package_count") != EXPECTED_PACKAGE_COUNT:
            raise ValueError(
                "resolution lock package count drifted"
            )
        if (
            lock.get("host_count")
            != EXPECTED_AUTHORITY_HOST_COUNT
        ):
            raise ValueError(
                "resolution lock authority-host count drifted"
            )

        runtime = lock.get("runtime")
        if not isinstance(runtime, dict):
            raise ValueError(
                "resolution lock runtime identity missing"
            )
        runtime_expected = {
            "python": EXPECTED_RUNTIME["python_major_minor"],
            "cuda_variant": EXPECTED_RUNTIME["cuda_variant"],
            "torch_cuda_version": (
                EXPECTED_RUNTIME["torch_cuda_version"]
            ),
            "torch_version": EXPECTED_RUNTIME["torch"],
            "vllm_distribution_version": (
                EXPECTED_RUNTIME["vllm"]
            ),
        }
        for key, expected_value in runtime_expected.items():
            if runtime.get(key) != expected_value:
                raise ValueError(
                    f"resolution lock runtime drifted: {key}"
                )

        raw_records = lock.get("records")
        if (
            not isinstance(raw_records, list)
            or len(raw_records) != EXPECTED_PACKAGE_COUNT
        ):
            raise ValueError(
                "resolution lock record set drifted"
            )

        seen_names: set[str] = set()
        seen_filenames: set[str] = set()
        lock_by_filename: dict[
            str,
            dict[str, object],
        ] = {}
        lock_records: list[dict[str, object]] = []

        for raw_record in raw_records:
            if not isinstance(raw_record, dict):
                raise ValueError(
                    "resolution lock record is not an object"
                )
            required = (
                "normalized_name",
                "version",
                "artifact_filename",
                "sha256",
            )
            if not all(
                isinstance(raw_record.get(key), str)
                for key in required
            ):
                raise ValueError(
                    "resolution lock record identity incomplete"
                )
            name = str(raw_record["normalized_name"])
            filename = str(
                raw_record["artifact_filename"]
            )
            sha256 = str(raw_record["sha256"])
            if name in seen_names:
                raise ValueError(
                    "duplicate locked distribution"
                )
            if filename in seen_filenames:
                raise ValueError(
                    "duplicate locked wheel filename"
                )
            if _SHA256_PATTERN.fullmatch(sha256) is None:
                raise ValueError("invalid locked SHA-256")
            if not filename.lower().endswith(".whl"):
                raise ValueError("non-wheel artifact locked")
            seen_names.add(name)
            seen_filenames.add(filename)
            lock_by_filename[filename] = raw_record
            lock_records.append(raw_record)

        wheel_files = tuple(
            sorted(
                (input_root / "wheels").glob("*.whl")
            )
        )
        observed_wheel_names = {
            path.name for path in wheel_files
        }
        if len(wheel_files) != EXPECTED_PACKAGE_COUNT:
            raise ValueError("wheel file count drifted")
        if observed_wheel_names != set(
            lock_by_filename
        ):
            raise ValueError(
                "wheel filename set drifted"
            )

        sha_manifest = load_object(
            input_root / "sha256_manifest.json"
        )
        if (
            sha_manifest.get("entry_count")
            != EXPECTED_SHA_MANIFEST_ENTRY_COUNT
        ):
            raise ValueError(
                "SHA manifest entry count drifted"
            )
        if (
            sha_manifest.get("wheel_entry_count")
            != EXPECTED_SHA_MANIFEST_WHEEL_COUNT
        ):
            raise ValueError(
                "SHA manifest wheel-entry count drifted"
            )
        if (
            sha_manifest.get("control_entry_count")
            != EXPECTED_SHA_MANIFEST_CONTROL_COUNT
        ):
            raise ValueError(
                "SHA manifest control-entry count drifted"
            )

        raw_entries = sha_manifest.get("entries")
        if (
            not isinstance(raw_entries, list)
            or len(raw_entries)
            != EXPECTED_SHA_MANIFEST_ENTRY_COUNT
        ):
            raise ValueError(
                "SHA manifest entry list drifted"
            )

        entry_paths: set[str] = set()
        wheel_entry_count = 0
        control_paths: set[str] = set()
        total_wheel_bytes = 0

        for raw_entry in raw_entries:
            if not isinstance(raw_entry, dict):
                raise ValueError(
                    "SHA manifest entry is not an object"
                )
            raw_path = raw_entry.get("path")
            expected_sha = raw_entry.get("sha256")
            expected_size = raw_entry.get("size_bytes")
            if not isinstance(raw_path, str):
                raise ValueError("SHA manifest path missing")
            if raw_path in entry_paths:
                raise ValueError(
                    "duplicate SHA manifest path"
                )
            entry_paths.add(raw_path)
            relative = validate_safe_relative_path(
                raw_path
            )
            if (
                not isinstance(expected_sha, str)
                or _SHA256_PATTERN.fullmatch(
                    expected_sha
                ) is None
            ):
                raise ValueError(
                    "invalid SHA manifest digest"
                )
            if (
                isinstance(expected_size, bool)
                or not isinstance(expected_size, int)
                or expected_size < 0
            ):
                raise ValueError(
                    "invalid SHA manifest size"
                )

            target = input_root.joinpath(
                *relative.parts
            )
            if not target.is_file():
                raise ValueError(
                    "manifest target missing"
                )
            if target.stat().st_size != expected_size:
                raise ValueError(
                    "manifest size mismatch"
                )
            if streaming_sha256(target) != expected_sha:
                raise ValueError("manifest SHA mismatch")

            if raw_path.startswith("wheels/"):
                wheel_entry_count += 1
                filename = relative.name
                locked = lock_by_filename.get(
                    filename
                )
                if locked is None:
                    raise ValueError(
                        "manifest contains unlocked wheel"
                    )
                if locked["sha256"] != expected_sha:
                    raise ValueError(
                        "manifest/lock wheel SHA mismatch"
                    )
                total_wheel_bytes += expected_size
            else:
                control_paths.add(raw_path)

        if wheel_entry_count != EXPECTED_PACKAGE_COUNT:
            raise ValueError(
                "manifest wheel-set count drifted"
            )
        if control_paths != set(
            EXPECTED_SHA_MANIFEST_CONTROL_PATHS
        ):
            raise ValueError(
                "manifest control path set drifted"
            )
        if total_wheel_bytes != EXPECTED_TOTAL_WHEEL_BYTES:
            raise ValueError(
                "total wheel bytes drifted"
            )

        requirements = (
            input_root / "requirements.lock.txt"
        ).read_text(encoding="utf-8")
        lock_tuple = tuple(lock_records)
        if requirements != expected_requirements(
            lock_tuple
        ):
            raise ValueError(
                "requirements.lock.txt does not reconstruct lock"
            )

        materialization_lock = (
            input_root / "materialization.lock.txt"
        ).read_text(encoding="utf-8")
        if (
            materialization_lock
            != expected_materialization_lock(lock_tuple)
        ):
            raise ValueError(
                "materialization.lock.txt does not reconstruct lock"
            )

        runtime_manifest = load_object(
            input_root / "runtime_manifest.json"
        )
        runtime_manifest_expected = {
            "exact_resolution_lock_sha256": (
                EXPECTED_RESOLUTION_LOCK_SHA256
            ),
            "locked_package_count": EXPECTED_PACKAGE_COUNT,
            "downloaded_package_count": (
                EXPECTED_PACKAGE_COUNT
            ),
            "authority_host_count": (
                EXPECTED_AUTHORITY_HOST_COUNT
            ),
            "observed_redirect_event_count": 1,
            "total_wheel_bytes": EXPECTED_TOTAL_WHEEL_BYTES,
            "dependency_resolution_performed": False,
            "package_installation_performed": False,
            "model_loads_performed": 0,
            "model_requests_performed": 0,
            "benchmark_trajectories_performed": 0,
            "credentials_used": False,
            "customer_data_used": False,
            "external_spend": 0,
        }
        for key, expected_value in (
            runtime_manifest_expected.items()
        ):
            if runtime_manifest.get(key) != expected_value:
                raise ValueError(
                    f"runtime manifest drifted: {key}"
                )

        receipt = load_object(
            input_root / "materialization_receipt.json"
        )
        for key, expected_value in (
            PRODUCER_RECEIPT_EXPECTED.items()
        ):
            if receipt.get(key) != expected_value:
                raise ValueError(
                    f"materialization receipt drifted: {key}"
                )

        if CONSUMER_CAPABILITY_POLICY != {
            "controlled_python_startup_required": True,
            "native_loader_provenance_required": True,
            "successful_native_import_alone_sufficient": False,
        }:
            raise ValueError(
                "consumer capability policy drifted"
            )

        observation = InputValidationObservation(
            input_root=input_root,
            lock_records=lock_tuple,
        )
        decision = ProbeDecision(
            status=ProbeStatus.PASSED
        )
    except Exception as exc:
        diagnostic_text = str(exc)
        decision = ProbeDecision(
            status=ProbeStatus.FAILED,
            failure_code=FailureCode.INPUT_VALIDATION_FAILED,
            detail="input validation failed",
        )

    duration_ms = int(
        (time.monotonic() - started) * 1000
    )
    evidence = project_local_evidence(
        role,
        decision,
        dependencies=(),
        started_at=started_at,
        duration_ms=duration_ms,
        diagnostic_text=diagnostic_text,
    )
    return ProbeOutcome(
        observation=observation,
        decision=decision,
        evidence=evidence,
    )


CONTROLLED_BOOTSTRAP = r"""
import site
import sys
import types
from pathlib import Path

expected_root = Path(sys.argv.pop(1)).resolve()
target_site = Path(sys.argv.pop(1)).resolve()
payload = sys.argv.pop(1)
expected_site = (
    expected_root
    / "lib"
    / f"python{sys.version_info.major}.{sys.version_info.minor}"
    / "site-packages"
).resolve()
if target_site != expected_site:
    raise RuntimeError(
        "target site-packages does not match target root"
    )


def sentinel(name):
    module = types.ModuleType(name)
    module.__file__ = f"<auragateway-suppressed-{name}>"
    return module


sys.modules["sitecustomize"] = sentinel("sitecustomize")
sys.modules["usercustomize"] = sentinel("usercustomize")
site.main()

cleaned = []
for value in sys.path:
    if not value:
        cleaned.append(value)
        continue
    path = Path(value).resolve()
    is_target = (
        path == target_site
        or target_site in path.parents
    )
    is_package_path = any(
        part in {"site-packages", "dist-packages"}
        for part in path.parts
    )
    if is_package_path and not is_target:
        continue
    cleaned.append(value)

if str(target_site) not in cleaned:
    cleaned.insert(0, str(target_site))
sys.path[:] = cleaned
sys.argv = [
    "<auragateway-controlled>",
    *sys.argv[1:],
]
exec(
    compile(
        payload,
        "<auragateway-controlled>",
        "exec",
    )
)
""".strip()


def controlled_probe_argv(
    payload: str,
) -> list[str]:
    target_python = TARGET_ROOT / "bin" / "python"
    return [
        str(target_python),
        "-S",
        "-c",
        CONTROLLED_BOOTSTRAP,
        str(TARGET_ROOT.resolve()),
        str(target_site_path().resolve()),
        payload,
    ]


STARTUP_PROBE_SCRIPT = r"""
import json
import os
import site
import sys
from pathlib import Path

external = []
for value in sys.path:
    if not value:
        continue
    path = Path(value).resolve()
    if any(
        part in {"site-packages", "dist-packages"}
        for part in path.parts
    ):
        if not (
            path == target_site
            or target_site in path.parents
        ):
            external.append(str(path))

print(
    json.dumps(
        {
            "prefix": str(Path(sys.prefix).resolve()),
            "base_prefix": str(
                Path(sys.base_prefix).resolve()
            ),
            "no_site_flag": sys.flags.no_site,
            "user_site_enabled": site.ENABLE_USER_SITE,
            "target_site_present": (
                str(target_site) in sys.path
            ),
            "external_package_paths": external,
            "sitecustomize_file": getattr(
                sys.modules["sitecustomize"],
                "__file__",
                None,
            ),
            "usercustomize_file": getattr(
                sys.modules["usercustomize"],
                "__file__",
                None,
            ),
            "pythonpath_present": (
                "PYTHONPATH" in os.environ
            ),
            "pythonhome_present": (
                "PYTHONHOME" in os.environ
            ),
            "ld_preload_present": (
                "LD_PRELOAD" in os.environ
            ),
            "python_no_user_site": os.environ.get(
                "PYTHONNOUSERSITE"
            ),
        },
        separators=(",", ":"),
        sort_keys=True,
    )
)
"""

NATIVE_INVENTORY_SCRIPT = r"""
import hashlib
import json
from pathlib import Path

vllm_root = target_site / "vllm"
required = sorted(
    vllm_root.glob("_C_stable_libtorch*.so")
)
legacy = (
    sorted(vllm_root.glob("_C.*.so"))
    + sorted(vllm_root.glob("_C.so"))
)
optional = sorted(
    path
    for pattern in (
        "_moe_C_stable_libtorch*.so",
        "_qutlass_C*.so",
    )
    for path in vllm_root.glob(pattern)
)
entries = []
for path in required:
    payload = path.read_bytes()
    entries.append(
        {
            "path": str(path.resolve()),
            "sha256": hashlib.sha256(
                payload
            ).hexdigest(),
            "size_bytes": len(payload),
        }
    )
print(
    json.dumps(
        {
            "required": entries,
            "legacy_c_candidates": [
                str(path.resolve())
                for path in legacy
            ],
            "optional_candidates": [
                str(path.resolve())
                for path in optional
            ],
        },
        separators=(",", ":"),
        sort_keys=True,
    )
)
"""

RUNTIME_PROVENANCE_SCRIPT = r"""
import importlib
import json
from pathlib import Path

import torch
import vllm

native = importlib.import_module(
    "vllm._C_stable_libtorch"
)
cuda_available = torch.cuda.is_available()
if cuda_available:
    _ = torch.cuda.get_device_name(0)

interesting = []
for line in Path("/proc/self/maps").read_text(
    encoding="utf-8"
).splitlines():
    fields = line.split()
    if len(fields) < 6:
        continue
    path = fields[-1]
    if not path.startswith("/"):
        continue
    name = Path(path).name
    if (
        "site-packages" in path
        or "dist-packages" in path
        or "/usr/local/cuda" in path
        or "/usr/local/nvidia" in path
        or name.startswith(
            (
                "libtorch",
                "libc10",
                "libcudart",
                "libnvJitLink",
                "libcusparse",
                "libcublas",
                "libcufft",
                "libcurand",
                "libcusolver",
                "libnccl",
                "libnvrtc",
                "libcuda",
            )
        )
    ):
        interesting.append(
            str(Path(path).resolve())
        )

print(
    json.dumps(
        {
            "native_module": (
                "vllm._C_stable_libtorch"
            ),
            "native_file": str(
                Path(native.__file__).resolve()
            ),
            "torch_file": str(
                Path(torch.__file__).resolve()
            ),
            "vllm_file": str(
                Path(vllm.__file__).resolve()
            ),
            "cuda_available": cuda_available,
            "loaded_paths": sorted(
                set(interesting)
            ),
        },
        separators=(",", ":"),
        sort_keys=True,
    )
)
"""

CUDA_PLATFORM_SCRIPT = r"""
import json
import sys

import torch
from vllm.platforms.cuda import CudaPlatform

CudaPlatform.import_kernels()
print(
    json.dumps(
        {
            "cuda_available": (
                torch.cuda.is_available()
            ),
            "cuda_device_count": (
                torch.cuda.device_count()
            ),
            "cuda_device_names": [
                torch.cuda.get_device_name(index)
                for index in range(
                    torch.cuda.device_count()
                )
            ],
            "required_native_module_loaded": (
                "vllm._C_stable_libtorch"
                in sys.modules
            ),
        },
        separators=(",", ":"),
        sort_keys=True,
    )
)
"""

DISTRIBUTION_SNAPSHOT_SCRIPT = r"""
import hashlib
import importlib.metadata
import json
import re

items = sorted(
    (
        re.sub(
            r"[-_.]+",
            "-",
            str(dist.metadata.get("Name", "")),
        ).lower(),
        dist.version,
    )
    for dist in importlib.metadata.distributions()
    if dist.metadata.get("Name")
)
encoded = json.dumps(
    items,
    ensure_ascii=True,
    separators=(",", ":"),
).encode("utf-8")
print(
    json.dumps(
        {
            "count": len(items),
            "sha256": hashlib.sha256(
                encoded
            ).hexdigest(),
        },
        separators=(",", ":"),
    )
)
"""

TARGET_IDENTITY_SCRIPT = r"""
import importlib.util
import json
import platform
import site
import sys
import sysconfig
from pathlib import Path

expected = Path(sys.argv[1]).resolve()
prefix = Path(sys.prefix).resolve()
base_prefix = Path(sys.base_prefix).resolve()
paths = sysconfig.get_paths()
config = (
    prefix / "pyvenv.cfg"
).read_text(encoding="utf-8").lower()

print(
    json.dumps(
        {
            "python": platform.python_version(),
            "prefix_matches_expected": (
                prefix == expected
            ),
            "base_prefix_differs": (
                base_prefix != prefix
            ),
            "user_site_enabled": (
                site.ENABLE_USER_SITE
            ),
            "purelib_within_prefix": (
                Path(
                    paths["purelib"]
                ).resolve().is_relative_to(
                    expected
                )
            ),
            "platlib_within_prefix": (
                Path(
                    paths["platlib"]
                ).resolve().is_relative_to(
                    expected
                )
            ),
            "system_site_packages_enabled": (
                "include-system-site-packages = true"
                in config
            ),
            "pip_present": (
                importlib.util.find_spec("pip")
                is not None
            ),
        },
        separators=(",", ":"),
    )
)
"""

TARGET_INVENTORY_SCRIPT = r"""
import importlib.metadata
import json
import re

items = sorted(
    (
        re.sub(
            r"[-_.]+",
            "-",
            str(dist.metadata.get("Name", "")),
        ).lower(),
        dist.version,
    )
    for dist in importlib.metadata.distributions()
    if dist.metadata.get("Name")
)
print(
    json.dumps(
        items,
        separators=(",", ":"),
    )
)
"""

PYTHON_RUNTIME_SCRIPT = r"""
import json
import platform

print(
    json.dumps(
        {
            "python": platform.python_version(),
        },
        separators=(",", ":"),
    )
)
"""

TORCH_FAMILY_SCRIPT = r"""
import json

import torch
import torchaudio
import torchvision

payload = {
    "torch": torch.__version__,
    "torchaudio": torchaudio.__version__,
    "torchvision": torchvision.__version__,
    "torch_cuda_version": torch.version.cuda,
    "cuda_available": torch.cuda.is_available(),
    "cuda_device_count": torch.cuda.device_count(),
    "cuda_device_names": [
        torch.cuda.get_device_name(index)
        for index in range(
            torch.cuda.device_count()
        )
    ],
}
print(
    json.dumps(
        payload,
        separators=(",", ":"),
    )
)
"""

TRANSFORMERS_SCRIPT = r"""
import json
import transformers

print(
    json.dumps(
        {
            "transformers": (
                transformers.__version__
            ),
        },
        separators=(",", ":"),
    )
)
"""

outcomes: dict[str, ProbeOutcome[object]] = {}
EVIDENCE_ROOT.mkdir(parents=True, exist_ok=False)

input_outcome = discover_and_validate_input()
outcomes["input_validation"] = input_outcome

outcomes["base_python_runtime"] = run_semantic_probe(
    "base_python_runtime",
    [
        sys.executable,
        "-c",
        (
            "import json,platform;"
            "print(json.dumps("
            "{'python':platform.python_version()},"
            "separators=(',',':')))"
        ),
    ],
    parse_python_version,
    validate_python_version,
    timeout=30.0,
    dependencies=(),
)

outcomes["base_pip_import"] = run_command_probe(
    "base_pip_import",
    [
        sys.executable,
        "-c",
        (
            "import json,pip;"
            "print(json.dumps("
            "{'pip':pip.__version__},"
            "separators=(',',':')))"
        ),
    ],
    timeout=30.0,
    dependencies=(),
)

outcomes[
    "base_distribution_snapshot_before"
] = run_semantic_probe(
    "base_distribution_snapshot_before",
    [
        sys.executable,
        "-c",
        DISTRIBUTION_SNAPSHOT_SCRIPT,
    ],
    parse_distribution_snapshot,
    validate_distribution_snapshot,
    timeout=60.0,
    dependencies=(),
)

outcomes["gpu_topology"] = run_semantic_probe(
    "gpu_topology",
    [
        "nvidia-smi",
        "--query-gpu="
        "index,name,uuid,memory.total,driver_version",
        "--format=csv,noheader,nounits",
    ],
    parse_gpu_topology,
    validate_gpu_topology,
    timeout=30.0,
    dependencies=(),
)

target_create_dependencies = (
    "input_validation",
    "base_python_runtime",
    "base_pip_import",
    "base_distribution_snapshot_before",
    "gpu_topology",
)
if dependencies_passed(
    outcomes,
    target_create_dependencies,
):
    if TARGET_ROOT.exists():
        shutil.rmtree(TARGET_ROOT)
    outcomes[
        "target_environment_creation"
    ] = run_command_probe(
        "target_environment_creation",
        [
            sys.executable,
            "-m",
            "venv",
            "--without-pip",
            str(TARGET_ROOT),
        ],
        timeout=120.0,
        dependencies=target_create_dependencies,
    )
else:
    outcomes[
        "target_environment_creation"
    ] = blocked_outcome(
        "target_environment_creation",
        target_create_dependencies,
        outcomes,
    )

target_python = TARGET_ROOT / "bin" / "python"

identity_dependencies = (
    "target_environment_creation",
)
if dependencies_passed(
    outcomes,
    identity_dependencies,
):
    outcomes[
        "target_runtime_identity_before_install"
    ] = run_semantic_probe(
        "target_runtime_identity_before_install",
        [
            str(target_python),
            "-c",
            TARGET_IDENTITY_SCRIPT,
            str(TARGET_ROOT),
        ],
        parse_target_identity,
        validate_target_identity,
        timeout=30.0,
        dependencies=identity_dependencies,
    )
else:
    outcomes[
        "target_runtime_identity_before_install"
    ] = blocked_outcome(
        "target_runtime_identity_before_install",
        identity_dependencies,
        outcomes,
    )

pip_support_dependencies = (
    "base_pip_import",
    "target_environment_creation",
)
if dependencies_passed(
    outcomes,
    pip_support_dependencies,
):
    outcomes[
        "base_pip_python_target_support"
    ] = run_command_probe(
        "base_pip_python_target_support",
        [
            sys.executable,
            "-m",
            "pip",
            "--isolated",
            "--disable-pip-version-check",
            "--python",
            str(TARGET_ROOT),
            "--version",
        ],
        timeout=30.0,
        dependencies=pip_support_dependencies,
    )
else:
    outcomes[
        "base_pip_python_target_support"
    ] = blocked_outcome(
        "base_pip_python_target_support",
        pip_support_dependencies,
        outcomes,
    )

install_dependencies = (
    "input_validation",
    "target_runtime_identity_before_install",
    "base_pip_python_target_support",
)
if dependencies_passed(
    outcomes,
    install_dependencies,
):
    assert input_outcome.observation is not None
    input_root = input_outcome.observation.input_root
    outcomes[
        "offline_hash_locked_install_via_base_pip"
    ] = run_command_probe(
        "offline_hash_locked_install_via_base_pip",
        [
            sys.executable,
            "-m",
            "pip",
            "--isolated",
            "--disable-pip-version-check",
            "--python",
            str(TARGET_ROOT),
            "install",
            "--no-index",
            "--no-cache-dir",
            "--no-deps",
            "--find-links",
            str(input_root / "wheels"),
            "--require-hashes",
            "-r",
            str(
                input_root
                / "requirements.lock.txt"
            ),
        ],
        timeout=1800.0,
        dependencies=install_dependencies,
    )
else:
    outcomes[
        "offline_hash_locked_install_via_base_pip"
    ] = blocked_outcome(
        "offline_hash_locked_install_via_base_pip",
        install_dependencies,
        outcomes,
    )

install_dependency = (
    "offline_hash_locked_install_via_base_pip",
)
if dependencies_passed(
    outcomes,
    install_dependency,
):
    assert input_outcome.observation is not None
    expected_inventory = tuple(
        sorted(
            (
                normalize_name(
                    str(item["normalized_name"])
                ),
                str(item["version"]),
            )
            for item
            in input_outcome.observation.lock_records
        )
    )
    outcomes[
        "target_distribution_inventory"
    ] = run_semantic_probe(
        "target_distribution_inventory",
        [
            str(target_python),
            "-c",
            TARGET_INVENTORY_SCRIPT,
        ],
        parse_distribution_inventory,
        lambda observation: (
            validate_distribution_inventory(
                observation,
                expected_items=expected_inventory,
            )
        ),
        timeout=120.0,
        dependencies=install_dependency,
    )
else:
    outcomes[
        "target_distribution_inventory"
    ] = blocked_outcome(
        "target_distribution_inventory",
        install_dependency,
        outcomes,
    )

if dependencies_passed(
    outcomes,
    install_dependency,
):
    outcomes[
        "target_dependency_check_via_base_pip"
    ] = run_command_probe(
        "target_dependency_check_via_base_pip",
        [
            sys.executable,
            "-m",
            "pip",
            "--isolated",
            "--disable-pip-version-check",
            "--python",
            str(TARGET_ROOT),
            "check",
        ],
        timeout=120.0,
        dependencies=install_dependency,
    )
else:
    outcomes[
        "target_dependency_check_via_base_pip"
    ] = blocked_outcome(
        "target_dependency_check_via_base_pip",
        install_dependency,
        outcomes,
    )

runtime_dependencies = (
    "target_distribution_inventory",
    "target_dependency_check_via_base_pip",
)
target_site = target_site_path().resolve()

if dependencies_passed(
    outcomes,
    runtime_dependencies,
):
    startup_environment = dict(os.environ)
    startup_environment.pop("PYTHONPATH", None)
    startup_environment.pop("PYTHONHOME", None)
    startup_environment.pop("LD_PRELOAD", None)
    startup_environment["PYTHONNOUSERSITE"] = "1"
    startup_environment[
        "PIP_DISABLE_PIP_VERSION_CHECK"
    ] = "1"
    startup_environment["PIP_NO_INDEX"] = "1"
    startup_environment["PIP_NO_CACHE_DIR"] = "1"
    startup_environment["HF_HUB_OFFLINE"] = "1"
    startup_environment[
        "TRANSFORMERS_OFFLINE"
    ] = "1"

    outcomes[
        "controlled_python_startup"
    ] = run_semantic_probe(
        "controlled_python_startup",
        controlled_probe_argv(
            STARTUP_PROBE_SCRIPT
        ),
        parse_controlled_startup,
        lambda observation: (
            validate_controlled_startup(
                observation,
                expected_root=TARGET_ROOT,
            )
        ),
        timeout=60.0,
        dependencies=runtime_dependencies,
        environment=startup_environment,
    )
else:
    outcomes[
        "controlled_python_startup"
    ] = blocked_outcome(
        "controlled_python_startup",
        runtime_dependencies,
        outcomes,
    )

startup_dependencies = (
    "controlled_python_startup",
)
if dependencies_passed(
    outcomes,
    startup_dependencies,
):
    outcomes[
        "target_native_inventory"
    ] = run_semantic_probe(
        "target_native_inventory",
        controlled_probe_argv(
            NATIVE_INVENTORY_SCRIPT
        ),
        parse_native_inventory,
        lambda observation: (
            validate_native_inventory(
                observation,
                target_vllm_root=(
                    target_site / "vllm"
                ),
            )
        ),
        timeout=60.0,
        dependencies=startup_dependencies,
        environment=startup_environment,
    )
else:
    outcomes[
        "target_native_inventory"
    ] = blocked_outcome(
        "target_native_inventory",
        startup_dependencies,
        outcomes,
    )

loader_dependencies = (
    "target_native_inventory",
)
loader_environment: dict[str, str] | None = None
if dependencies_passed(
    outcomes,
    loader_dependencies,
):
    loader_started = time.monotonic()
    loader_started_at = datetime.now(
        UTC
    ).isoformat(timespec="seconds")
    (
        loader_environment,
        loader_observation,
        loader_decision,
    ) = build_canonical_loader_environment(
        target_site
    )
    loader_evidence = project_local_evidence(
        "canonical_loader_environment",
        loader_decision,
        dependencies=loader_dependencies,
        started_at=loader_started_at,
        duration_ms=int(
            (time.monotonic() - loader_started)
            * 1000
        ),
        diagnostic_text=canonical_json(
            {
                "target_nvidia_library_count": (
                    loader_observation
                    .target_nvidia_library_count
                ),
                "target_torch_library": str(
                    loader_observation
                    .target_torch_library
                ),
                "real_driver_directory": str(
                    loader_observation
                    .real_driver_directory
                ),
                "ld_preload_absent": (
                    loader_observation
                    .ld_preload_absent
                ),
                "pythonpath_absent": (
                    loader_observation
                    .pythonpath_absent
                ),
                "pythonhome_absent": (
                    loader_observation
                    .pythonhome_absent
                ),
                "prohibited_path_present": (
                    loader_observation
                    .prohibited_path_present
                ),
            }
        ),
    )
    outcomes[
        "canonical_loader_environment"
    ] = ProbeOutcome(
        observation=loader_observation,
        decision=loader_decision,
        evidence=loader_evidence,
    )
else:
    outcomes[
        "canonical_loader_environment"
    ] = blocked_outcome(
        "canonical_loader_environment",
        loader_dependencies,
        outcomes,
    )

controlled_runtime_dependencies = (
    "controlled_python_startup",
    "canonical_loader_environment",
)

if (
    dependencies_passed(
        outcomes,
        controlled_runtime_dependencies,
    )
    and loader_environment is not None
):
    outcomes["python_runtime"] = run_semantic_probe(
        "python_runtime",
        controlled_probe_argv(
            PYTHON_RUNTIME_SCRIPT
        ),
        parse_python_version,
        validate_python_version,
        timeout=30.0,
        dependencies=controlled_runtime_dependencies,
        environment=loader_environment,
    )
else:
    outcomes["python_runtime"] = blocked_outcome(
        "python_runtime",
        controlled_runtime_dependencies,
        outcomes,
    )

if (
    dependencies_passed(
        outcomes,
        controlled_runtime_dependencies,
    )
    and loader_environment is not None
):
    outcomes[
        "torch_family_runtime"
    ] = run_semantic_probe(
        "torch_family_runtime",
        controlled_probe_argv(
            TORCH_FAMILY_SCRIPT
        ),
        parse_torch_family,
        validate_torch_family,
        timeout=180.0,
        dependencies=controlled_runtime_dependencies,
        environment=loader_environment,
    )
else:
    outcomes[
        "torch_family_runtime"
    ] = blocked_outcome(
        "torch_family_runtime",
        controlled_runtime_dependencies,
        outcomes,
    )

if (
    dependencies_passed(
        outcomes,
        controlled_runtime_dependencies,
    )
    and loader_environment is not None
):
    outcomes[
        "transformers_runtime"
    ] = run_semantic_probe(
        "transformers_runtime",
        controlled_probe_argv(
            TRANSFORMERS_SCRIPT
        ),
        lambda raw: parse_version_field(
            raw,
            key="transformers",
            package="transformers",
        ),
        lambda observation: (
            validate_exact_version(
                observation,
                expected=EXPECTED_RUNTIME[
                    "transformers"
                ],
            )
        ),
        timeout=60.0,
        dependencies=controlled_runtime_dependencies,
        environment=loader_environment,
    )
else:
    outcomes[
        "transformers_runtime"
    ] = blocked_outcome(
        "transformers_runtime",
        controlled_runtime_dependencies,
        outcomes,
    )

if (
    dependencies_passed(
        outcomes,
        controlled_runtime_dependencies,
    )
    and loader_environment is not None
):
    outcomes[
        "triton_distribution"
    ] = run_semantic_probe(
        "triton_distribution",
        controlled_probe_argv(
            (
                "import importlib.metadata,json;"
                "print(json.dumps("
                "{'triton':"
                "importlib.metadata.version('triton')},"
                "separators=(',',':')))"
            )
        ),
        lambda raw: parse_version_field(
            raw,
            key="triton",
            package="triton",
        ),
        lambda observation: (
            validate_exact_version(
                observation,
                expected=EXPECTED_RUNTIME[
                    "triton"
                ],
            )
        ),
        timeout=30.0,
        dependencies=controlled_runtime_dependencies,
        environment=loader_environment,
    )
else:
    outcomes[
        "triton_distribution"
    ] = blocked_outcome(
        "triton_distribution",
        controlled_runtime_dependencies,
        outcomes,
    )

if (
    dependencies_passed(
        outcomes,
        controlled_runtime_dependencies,
    )
    and loader_environment is not None
):
    outcomes[
        "vllm_distribution"
    ] = run_semantic_probe(
        "vllm_distribution",
        controlled_probe_argv(
            (
                "import importlib.metadata,json;"
                "print(json.dumps("
                "{'vllm':"
                "importlib.metadata.version('vllm')},"
                "separators=(',',':')))"
            )
        ),
        lambda raw: parse_version_field(
            raw,
            key="vllm",
            package="vllm_distribution",
        ),
        lambda observation: (
            validate_exact_version(
                observation,
                expected=EXPECTED_RUNTIME["vllm"],
            )
        ),
        timeout=30.0,
        dependencies=controlled_runtime_dependencies,
        environment=loader_environment,
    )
else:
    outcomes[
        "vllm_distribution"
    ] = blocked_outcome(
        "vllm_distribution",
        controlled_runtime_dependencies,
        outcomes,
    )

vllm_module_dependencies = (
    "torch_family_runtime",
    "transformers_runtime",
    "triton_distribution",
    "vllm_distribution",
)
if (
    dependencies_passed(
        outcomes,
        vllm_module_dependencies,
    )
    and loader_environment is not None
):
    outcomes["vllm_module"] = run_semantic_probe(
        "vllm_module",
        controlled_probe_argv(
            (
                "import json,vllm;"
                "print(json.dumps("
                "{'vllm':vllm.__version__},"
                "separators=(',',':')))"
            )
        ),
        lambda raw: parse_version_field(
            raw,
            key="vllm",
            package="vllm_module",
        ),
        lambda observation: (
            validate_exact_version(
                observation,
                expected=(
                    EXPECTED_VLLM_MODULE_VERSION
                ),
            )
        ),
        timeout=180.0,
        dependencies=vllm_module_dependencies,
        environment=loader_environment,
    )
else:
    outcomes["vllm_module"] = blocked_outcome(
        "vllm_module",
        vllm_module_dependencies,
        outcomes,
    )

static_dependencies = (
    "target_native_inventory",
    "canonical_loader_environment",
)
if (
    dependencies_passed(
        outcomes,
        static_dependencies,
    )
    and loader_environment is not None
):
    native_inventory_outcome = outcomes[
        "target_native_inventory"
    ]
    native_inventory_observation = (
        native_inventory_outcome.observation
    )
    assert isinstance(
        native_inventory_observation,
        NativeInventoryObservation,
    )
    native_path = (
        native_inventory_observation
        .required[0]
        .path
        .resolve()
    )
    outcomes[
        "native_linker_static_provenance"
    ] = run_semantic_probe(
        "native_linker_static_provenance",
        ["ldd", str(native_path)],
        parse_ldd,
        lambda observation: (
            validate_native_linker(
                observation,
                target_site=target_site,
            )
        ),
        timeout=60.0,
        dependencies=static_dependencies,
        environment=loader_environment,
    )
else:
    outcomes[
        "native_linker_static_provenance"
    ] = blocked_outcome(
        "native_linker_static_provenance",
        static_dependencies,
        outcomes,
    )

native_dependencies = (
    "vllm_module",
    "native_linker_static_provenance",
)
if (
    dependencies_passed(
        outcomes,
        native_dependencies,
    )
    and loader_environment is not None
):
    outcomes[
        "vllm_native_extension"
    ] = run_semantic_probe(
        "vllm_native_extension",
        controlled_probe_argv(
            (
                "import importlib,json;"
                "module=importlib.import_module("
                "'vllm._C_stable_libtorch');"
                "print(json.dumps("
                "{'native_extension':"
                "'vllm._C_stable_libtorch',"
                "'file':module.__file__},"
                "separators=(',',':')))"
            )
        ),
        parse_native_extension,
        lambda observation: (
            validate_native_extension(
                observation,
                target_vllm_root=(
                    target_site / "vllm"
                ),
            )
        ),
        timeout=180.0,
        dependencies=native_dependencies,
        environment=loader_environment,
    )
else:
    outcomes[
        "vllm_native_extension"
    ] = blocked_outcome(
        "vllm_native_extension",
        native_dependencies,
        outcomes,
    )

provenance_dependencies = (
    "vllm_native_extension",
    "torch_family_runtime",
)
if (
    dependencies_passed(
        outcomes,
        provenance_dependencies,
    )
    and loader_environment is not None
):
    outcomes[
        "native_runtime_provenance"
    ] = run_semantic_probe(
        "native_runtime_provenance",
        controlled_probe_argv(
            RUNTIME_PROVENANCE_SCRIPT
        ),
        parse_native_runtime_provenance,
        lambda observation: (
            validate_native_runtime_provenance(
                observation,
                target_site=target_site,
            )
        ),
        timeout=180.0,
        dependencies=provenance_dependencies,
        environment=loader_environment,
    )
else:
    outcomes[
        "native_runtime_provenance"
    ] = blocked_outcome(
        "native_runtime_provenance",
        provenance_dependencies,
        outcomes,
    )

platform_dependencies = (
    "native_runtime_provenance",
    "vllm_native_extension",
)
if (
    dependencies_passed(
        outcomes,
        platform_dependencies,
    )
    and loader_environment is not None
):
    outcomes[
        "cuda_platform_capability"
    ] = run_semantic_probe(
        "cuda_platform_capability",
        controlled_probe_argv(
            CUDA_PLATFORM_SCRIPT
        ),
        parse_cuda_platform,
        validate_cuda_platform,
        timeout=180.0,
        dependencies=platform_dependencies,
        environment=loader_environment,
    )
else:
    outcomes[
        "cuda_platform_capability"
    ] = blocked_outcome(
        "cuda_platform_capability",
        platform_dependencies,
        outcomes,
    )

snapshot_after_dependencies = (
    "base_distribution_snapshot_before",
)
snapshot_after = run_semantic_probe(
    "base_distribution_snapshot_after",
    [
        sys.executable,
        "-c",
        DISTRIBUTION_SNAPSHOT_SCRIPT,
    ],
    parse_distribution_snapshot,
    validate_distribution_snapshot,
    timeout=60.0,
    dependencies=snapshot_after_dependencies,
)
if (
    snapshot_after.decision.status
    == ProbeStatus.PASSED
):
    before_observation = outcomes[
        "base_distribution_snapshot_before"
    ].observation
    after_observation = snapshot_after.observation
    if (
        not isinstance(
            before_observation,
            DistributionSnapshotObservation,
        )
        or not isinstance(
            after_observation,
            DistributionSnapshotObservation,
        )
        or before_observation != after_observation
    ):
        raw = RawProbeExecution(
            command_role=(
                "base_distribution_snapshot_after"
            ),
            returncode=0,
            timed_out=False,
            duration_ms=(
                snapshot_after.evidence.duration_ms
            ),
            started_at=(
                snapshot_after.evidence.started_at
                or ""
            ),
            stdout="",
            stderr="",
        )
        decision = ProbeDecision(
            status=ProbeStatus.FAILED,
            failure_code=(
                FailureCode.SEMANTIC_CONTRACT_FAILED
            ),
            detail=(
                "base distribution snapshot changed"
            ),
        )
        snapshot_after = ProbeOutcome(
            observation=after_observation,
            decision=decision,
            evidence=project_evidence(
                raw,
                decision,
                dependencies=(
                    snapshot_after_dependencies
                ),
            ),
        )
outcomes[
    "base_distribution_snapshot_after"
] = snapshot_after

for role in REQUIRED_ROLES:
    if role not in outcomes:
        decision = ProbeDecision(
            status=ProbeStatus.NOT_EXECUTED,
            detail="role was not scheduled",
        )
        outcomes[role] = ProbeOutcome(
            observation=None,
            decision=decision,
            evidence=project_local_evidence(
                role,
                decision,
                dependencies=(),
                started_at=None,
                duration_ms=0,
            ),
        )

required_statuses = {
    role: outcomes[role].decision.status.value
    for role in REQUIRED_ROLES
}
failed_required_roles = [
    role
    for role in REQUIRED_ROLES
    if (
        outcomes[role].decision.status
        != ProbeStatus.PASSED
    )
]

install_outcome = outcomes[
    "offline_hash_locked_install_via_base_pip"
]
package_installation_started = (
    install_outcome.decision.status
    in {
        ProbeStatus.PASSED,
        ProbeStatus.FAILED,
    }
)

summary = {
    "schema_version": "1.0.0",
    "notebook_name": NOTEBOOK_NAME,
    "requested_kaggle_title": REQUESTED_KAGGLE_TITLE,
    "expected_materializer_script_version_id": (
        EXPECTED_MATERIALIZER_SCRIPT_VERSION_ID
    ),
    "predecessor_v4_saved_version_id": (
        PREDECESSOR_V4_SAVED_VERSION_ID
    ),
    "predecessor_v4_failure_class": (
        PREDECESSOR_V4_FAILURE_CLASS
    ),
    "predecessor_v4_failure_code": (
        PREDECESSOR_V4_FAILURE_CODE
    ),
    "predecessor_v4_evidence_zip_sha256": (
        PREDECESSOR_V4_EVIDENCE_ZIP_SHA256
    ),
    "required_native_module": REQUIRED_NATIVE_MODULE,
    "offline_compatibility_status": (
        "PASSED_PENDING_REPOSITORY_ACCEPTANCE"
        if not failed_required_roles
        else "FAILED_PENDING_REVIEW"
    ),
    "required_role_statuses": required_statuses,
    "failed_required_roles": failed_required_roles,
    "locked_package_count": EXPECTED_PACKAGE_COUNT,
    "validated_manifest_entry_count": (
        EXPECTED_SHA_MANIFEST_ENTRY_COUNT
    ),
    "total_wheel_bytes": EXPECTED_TOTAL_WHEEL_BYTES,
    "package_installation_started": (
        package_installation_started
    ),
    "package_installation_performed": (
        install_outcome.decision.status
        == ProbeStatus.PASSED
    ),
    "dependency_resolution_performed": False,
    "internet_required": False,
    "model_loads_performed": 0,
    "model_requests_performed": 0,
    "worker_startups_performed": 0,
    "benchmark_trajectories_performed": 0,
    "credentials_used": False,
    "customer_data_used": False,
    "external_spend": 0,
    "qualification_claimed": False,
    "raw_probe_execution_transient": True,
    "raw_streams_persisted": False,
    "semantic_decisions_reading_stdout_excerpt": 0,
    "semantic_decisions_reading_stderr_excerpt": 0,
    "lossy_transformations_before_semantic_decision": 0,
    "truncation_before_semantic_decision": 0,
    "evidence_projection_terminal": True,
    "exact_runtime_offline_verified": False,
    "p5_p6_exact_runtime_requalified": False,
    "runtime_execution_authorized": False,
    "pilot_execution_authorized": False,
    "final_measured_abc_execution_authorized": False,
}

records = {
    role: outcomes[role].evidence.to_dict()
    for role in REQUIRED_ROLES
}

input_evidence = {
    "schema_version": "1.0.0",
    "input_directory_name": INPUT_DIRECTORY_NAME,
    "input_validation": (
        outcomes["input_validation"]
        .evidence
        .to_dict()
    ),
    "expected_resolution_lock_sha256": (
        EXPECTED_RESOLUTION_LOCK_SHA256
    ),
    "expected_control_hashes": EXPECTED_CONTROL_HASHES,
    "expected_package_count": EXPECTED_PACKAGE_COUNT,
    "expected_manifest_entry_count": (
        EXPECTED_SHA_MANIFEST_ENTRY_COUNT
    ),
    "expected_total_wheel_bytes": (
        EXPECTED_TOTAL_WHEEL_BYTES
    ),
}

write_json(
    EVIDENCE_ROOT / "input_validation.json",
    input_evidence,
)
write_json(
    EVIDENCE_ROOT / "probe_records.json",
    records,
)
write_json(
    EVIDENCE_ROOT / "verification_summary.json",
    summary,
)

evidence_members = (
    "input_validation.json",
    "probe_records.json",
    "verification_summary.json",
)
evidence_manifest_entries = []
for name in evidence_members:
    path = EVIDENCE_ROOT / name
    evidence_manifest_entries.append(
        {
            "path": name,
            "sha256": streaming_sha256(path),
            "size_bytes": path.stat().st_size,
        }
    )

evidence_manifest = {
    "schema_version": "1.0.0",
    "entry_count": len(evidence_manifest_entries),
    "entries": evidence_manifest_entries,
}
write_json(
    EVIDENCE_ROOT / "evidence_manifest.json",
    evidence_manifest,
)

with zipfile.ZipFile(
    OUTPUT_ZIP,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as archive:
    for name in (
        *evidence_members,
        "evidence_manifest.json",
    ):
        archive.write(
            EVIDENCE_ROOT / name,
            arcname=name,
        )

print(
    canonical_json(
        {
            **summary,
            "evidence_zip": OUTPUT_ZIP.name,
            "evidence_zip_sha256": (
                streaming_sha256(OUTPUT_ZIP)
            ),
            "upload_only_this_file": True,
            "preserve_saved_version": True,
        }
    )
)


{"benchmark_trajectories_performed":0,"credentials_used":false,"customer_data_used":false,"dependency_resolution_performed":false,"evidence_projection_terminal":true,"evidence_zip":"auragateway_preflight_v3_exact_runtime_offline_compatibility_evidence_v5.zip","evidence_zip_sha256":"59db662ec4722f0db57e2e57449577e28a3efbaeda155db8ed6f245774bff06a","exact_runtime_offline_verified":false,"expected_materializer_script_version_id":341083505,"external_spend":0,"failed_required_roles":[],"final_measured_abc_execution_authorized":false,"internet_required":false,"locked_package_count":196,"lossy_transformations_before_semantic_decision":0,"model_loads_performed":0,"model_requests_performed":0,"notebook_name":"auragateway-preflight-v3-exact-runtime-offline-compatibility-v5","offline_compatibility_status":"PASSED_PENDING_REPOSITORY_ACCEPTANCE","p5_p6_exact_runtime_requalified":false,"package_installation_performed":true,"package_installation_started":true,"pilot_execution_authorized":false,"prede